# TVG 자료 확인
전체 세포 사진과 원본 설명·위치 주석을 확인합니다. 질환 학습 노트북이 아닙니다.


In [ ]:
from pathlib import Path
import pandas as pd
import json

BASE = Path(r"C:\chromosome_ai\tvg_stage2")
assert (BASE / "case_catalog.csv").is_file(), "압축을 푼 폴더 위치를 확인해줘."
catalog = pd.read_csv(BASE / "case_catalog.csv", keep_default_na=False)
paired = catalog[catalog["acquisition_status"] == "paired"].copy()
print("원본 이상 설명 행:", len(catalog))
print("사진·주석 확보:", len(paired))
display(paired.groupby("source_category").size().rename("사진 수").to_frame())
display(catalog.head())


In [ ]:
missing = []
for _, row in paired.iterrows():
    for column in ["image_path", "annotation_path"]:
        if not (BASE / row[column]).is_file():
            missing.append(row[column])
print("로컬 누락 파일:", len(missing))
assert not missing, missing[:5]
report = json.loads((BASE / "audit_report.json").read_text(encoding="utf-8"))
print("CSV 총수 / XML 객체 수 불일치:", len(report["csv_total_xml_object_count_mismatches"]))
print("질환명 정답 제공:", report["disease_ground_truth_available"])
print("환자 그룹 정보 제공:", report["patient_grouping_available"])


## 원본 주석으로 21번 위치 확인
아래 상자는 데이터 제공자의 정답 주석입니다. AI 검출 결과나 질환 확진 결과가 아닙니다.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import xml.etree.ElementTree as ET

choices = paired[
    (paired["source_category"] == "number_abnormalities") &
    (paired["source_affected_chromosome_description"].astype(str) == "21") &
    (paired["source_total_chromosomes"].astype(str) == "47")
]
row = choices.iloc[0]
annotation = ET.parse(BASE / row["annotation_path"])
with Image.open(BASE / row["image_path"]) as source:
    image = source.convert("RGB")
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(image)
selected = 0
for obj in annotation.findall("object"):
    if obj.findtext("name") != "G21":
        continue
    x1, y1, x2, y2 = [float(obj.findtext("bndbox/" + key)) for key in ["xmin", "ymin", "xmax", "ymax"]]
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="#00b894", linewidth=2))
    ax.text(x1, max(0, y1-4), "chr21 annotation", color="#006b52", fontsize=9)
    selected += 1
ax.set_title(str(row["filename"]) + " | Source annotations")
ax.axis("off")
plt.show()
print("원본 CSV의 총수:", row["source_total_chromosomes"])
print("원본 CSV의 이상 관련 번호:", row["source_affected_chromosome_description"])
print("XML에 표시된 21번 객체 수:", selected)
print("질환명 정답은 제공되지 않았어. 현재 화면은 원본 주석 확인용이야.")


In [ ]:
import re
from PIL import ImageOps

case_id = Path(row["filename"]).stem
crop_dir = BASE / "derived" / "number_crops" / case_id
crop_dir.mkdir(parents=True, exist_ok=True)

crop_records = []

for index, obj in enumerate(annotation.findall("object")):
    source_label = obj.findtext("name")

    # 원본 주석 이름 → 염색체 번호
    if source_label in ["X", "Y"]:
        chromosome = "chr" + source_label
    else:
        match = re.fullmatch(r"[A-G](\d+)", source_label or "")
        if not match:
            continue
        chromosome = "chr" + match.group(1)

    box = tuple(
        int(obj.findtext("bndbox/" + key))
        for key in ["xmin", "ymin", "xmax", "ymax"]
    )

    # 원본 위치에서 자르고, 비율을 유지하도록 흰 여백 추가
    cropped = image.crop(box)
    side = max(cropped.size) + 12
    prepared = ImageOps.pad(
        cropped,
        (side, side),
        color="white",
    )

    output_path = crop_dir / f"{index:02d}_{chromosome}.png"
    prepared.save(output_path)

    crop_records.append({
        "source_image": row["filename"],
        "object_index": index,
        "chromosome_ground_truth": chromosome,
        "crop_path": str(output_path),
    })

crop_table = pd.DataFrame(crop_records)
crop_table.to_csv(
    crop_dir / "crop_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장한 염색체 조각:", len(crop_table), "개")
print("21번 정답 조각:", (crop_table["chromosome_ground_truth"] == "chr21").sum(), "개")
print("저장 위치:", crop_dir)

In [ ]:
import argparse
import joblib
import timm
import torch
import numpy as np
from torchvision import transforms
from tqdm.auto import tqdm

project_dir = Path(r"C:\chromosome_ai")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 저장된 번호 분류기
tvg_number_bundle = joblib.load(
    project_dir / "models/chroma_number_task_linear_v1/classifier.joblib"
)
tvg_number_classifier = tvg_number_bundle["classifier"]
tvg_chromosome_names = tvg_number_bundle["chromosome_names"]

# 2. 현재 노트북에 번호 인코더가 없으면 복원
if "tvg_number_encoder" not in globals():
    with torch.serialization.safe_globals([argparse.Namespace]):
        checkpoint = torch.load(
            project_dir / "pretrained/CHROMA/CHROMA-num_ab.pth",
            map_location="cpu",
            weights_only=True,
            mmap=True,
        )

    encoder_state = {
        name: value
        for name, value in checkpoint["model"].items()
        if not name.startswith("head.")
    }

    tvg_number_encoder = timm.create_model(
        "vit_large_patch16_224",
        pretrained=False,
        num_classes=0,
        global_pool="avg",
    )
    tvg_number_encoder.load_state_dict(encoder_state, strict=True)
    tvg_number_encoder.requires_grad_(False)
    tvg_number_encoder = tvg_number_encoder.to(device)
    tvg_number_encoder.eval()

    del checkpoint, encoder_state

print("번호 모델 준비 완료")

# 3. 학습 당시 전처리
tvg_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

def load_crop_tensor(path):
    with Image.open(path) as source:
        return tvg_transform(source.convert("RGB"))

# 4. 네 장씩 특징 추출 후 번호 예측
feature_batches = []
tvg_number_encoder.eval()

with torch.inference_mode():
    for start in tqdm(
        range(0, len(crop_table), 4),
        desc="번호 예측",
    ):
        paths = crop_table["crop_path"].iloc[start:start + 4]
        inputs = torch.stack([
            load_crop_tensor(path) for path in paths
        ]).to(device)

        features = tvg_number_encoder(inputs)
        feature_batches.append(features.cpu().numpy())

crop_features = np.concatenate(feature_batches, axis=0)
predicted_ids = tvg_number_classifier.predict(crop_features)

crop_predictions = crop_table.copy()
crop_predictions["predicted_chromosome"] = [
    tvg_chromosome_names[int(index)] for index in predicted_ids
]
crop_predictions["correct"] = (
    crop_predictions["chromosome_ground_truth"]
    == crop_predictions["predicted_chromosome"]
)

# 5. 번호 정확도와 21번 오분류 확인
actual_21 = crop_predictions["chromosome_ground_truth"] == "chr21"
predicted_21 = crop_predictions["predicted_chromosome"] == "chr21"

print(
    "\n이 사진의 조각별 번호 정확도:",
    f"{crop_predictions['correct'].mean():.2%}",
)
print("실제 21번:", int(actual_21.sum()), "개")
print("AI가 21번으로 예측:", int(predicted_21.sum()), "개")
print("21번을 정확히 찾음:", int((actual_21 & predicted_21).sum()), "개")
print("다른 번호를 21번으로 오분류:", int((~actual_21 & predicted_21).sum()), "개")
print("실제 21번을 놓침:", int((actual_21 & ~predicted_21).sum()), "개")

display(
    crop_predictions.loc[
        ~crop_predictions["correct"],
        ["object_index", "chromosome_ground_truth", "predicted_chromosome"],
    ].head(10)
)

crop_predictions.to_csv(
    crop_dir / "number_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# 실제 학습에 사용한 사진 목록
reference_train = pd.read_csv(
    r"C:\chromosome_ai\splits\chroma_demo_v1\train.csv"
)

def show_model_input(path):
    with Image.open(path) as source:
        tensor = tvg_transform(source.convert("RGB"))

    # 화면 표시를 위해 정규화만 되돌림
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    pixels = (tensor * std + mean).clamp(0, 1)

    return pixels.permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for row_index, chromosome in enumerate(["chr1", "chr21"]):
    reference_path = reference_train.loc[
        reference_train["chromosome"] == chromosome, "path"
    ].iloc[0]

    tvg_path = crop_table.loc[
        crop_table["chromosome_ground_truth"] == chromosome,
        "crop_path",
    ].iloc[0]

    for column, (path, title) in enumerate([
        (reference_path, f"Training input: {chromosome}"),
        (tvg_path, f"TVG current input: {chromosome}"),
    ]):
        axes[row_index, column].imshow(show_model_input(path))
        axes[row_index, column].set_title(title)
        axes[row_index, column].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# 원본 사진을 다시 열기
with Image.open(BASE / row["image_path"]) as source:
    original_image = source.convert("RGB")

objects = annotation.findall("object")
raw_crops = []

for obj in objects:
    box = tuple(
        int(obj.findtext(f"bndbox/{key}"))
        for key in ["xmin", "ymin", "xmax", "ymax"]
    )
    raw_crops.append(original_image.crop(box))

# 모든 조각에 공통으로 사용할 바탕 크기
canvas_size = max(
    max(crop.size) for crop in raw_crops
) + 12

fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for row_index, chromosome in enumerate(["chr1", "chr21"]):
    record = crop_table.loc[
        crop_table["chromosome_ground_truth"] == chromosome
    ].iloc[0]

    object_index = int(record["object_index"])
    raw_crop = raw_crops[object_index]

    # 확대하지 않고, 공통 크기의 흰 바탕 가운데에 배치
    canvas = Image.new("RGB", (canvas_size, canvas_size), "white")
    position = (
        (canvas_size - raw_crop.width) // 2,
        (canvas_size - raw_crop.height) // 2,
    )
    canvas.paste(raw_crop, position)

    with Image.open(record["crop_path"]) as previous:
        axes[row_index, 0].imshow(previous.convert("RGB"))

    axes[row_index, 0].set_title(f"Previous input: {chromosome}")
    axes[row_index, 1].imshow(canvas)
    axes[row_index, 1].set_title(f"Shared canvas: {chromosome}")

    for ax in axes[row_index]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

shared_table = crop_table.copy().reset_index(drop=True)
shared_inputs = []

for _, record in shared_table.iterrows():
    crop = raw_crops[int(record["object_index"])]

    canvas = Image.new(
        "RGB", (canvas_size, canvas_size), "white"
    )
    canvas.paste(
        crop,
        (
            (canvas_size - crop.width) // 2,
            (canvas_size - crop.height) // 2,
        ),
    )

    shared_inputs.append(tvg_transform(canvas))

feature_batches = []
tvg_number_encoder.eval()

with torch.inference_mode():
    for start in tqdm(
        range(0, len(shared_inputs), 4),
        desc="공통 바탕 번호 예측"
    ):
        batch = torch.stack(
            shared_inputs[start:start + 4]
        ).to(device)

        features = tvg_number_encoder(batch)
        feature_batches.append(features.cpu().numpy())

shared_features = np.concatenate(feature_batches, axis=0)
predicted_ids = tvg_number_classifier.predict(shared_features)

shared_table["predicted_chromosome"] = [
    tvg_chromosome_names[int(i)] for i in predicted_ids
]

correct = (
    shared_table["chromosome_ground_truth"]
    == shared_table["predicted_chromosome"]
)
actual_21 = shared_table["chromosome_ground_truth"].eq("chr21")
predicted_21 = shared_table["predicted_chromosome"].eq("chr21")

print(f"번호 정확도: {correct.mean():.2%}")
print(f"번호 정답: {correct.sum()}/{len(shared_table)}개")
print(f"실제 21번: {actual_21.sum()}개")
print(f"AI가 21번으로 예측: {predicted_21.sum()}개")
print(f"21번을 정확히 찾음: {(actual_21 & predicted_21).sum()}개")
print(f"다른 번호를 21번으로 오분류: {(~actual_21 & predicted_21).sum()}개")
print(f"실제 21번을 놓침: {(actual_21 & ~predicted_21).sum()}개")

display(shared_table[
    ["object_index", "chromosome_ground_truth",
     "predicted_chromosome"]
])

In [ ]:
from collections import Counter
import xml.etree.ElementTree as ET
import re
import pandas as pd

object_counts = Counter()
image_counts = Counter()
unknown_labels = Counter()

for _, case in paired.iterrows():
    root = ET.parse(BASE / case["annotation_path"]).getroot()
    present = set()

    for obj in root.findall("object"):
        label = (obj.findtext("name") or "").strip()

        if label in ("X", "Y"):
            chromosome = f"chr{label}"
        else:
            match = re.fullmatch(r"[A-G](\d+)", label)
            if not match:
                unknown_labels[label] += 1
                continue
            chromosome = f"chr{int(match.group(1))}"

        object_counts[chromosome] += 1
        present.add(chromosome)

    image_counts.update(present)

order = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

summary = pd.DataFrame({
    "염색체": order,
    "정답 상자 수": [object_counts[c] for c in order],
    "포함된 원본 사진 수": [image_counts[c] for c in order],
})

print("확인한 원본 사진:", len(paired), "장")
print("전체 정답 상자:", sum(object_counts.values()), "개")
print("해석하지 못한 라벨:", dict(unknown_labels))
display(summary)

In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd

cases = paired.copy()

csv_total = pd.to_numeric(
    cases["source_total_chromosomes"], errors="coerce"
)
xml_total = pd.to_numeric(
    cases["xml_object_count"], errors="coerce"
)

# 총수 불일치·확인 불가 사진 제외
needs_review = (
    csv_total.isna()
    | xml_total.isna()
    | csv_total.ne(xml_total)
)

# 이미 예측 결과를 보며 조정한 사진 제외
used_demo = cases["filename"].eq("1052521.jpg")

excluded = cases.loc[needs_review | used_demo].copy()
eligible = cases.loc[~(needs_review | used_demo)].copy()

# 개수 이상 / 구조 이상 사진 비율을 대략 유지
train_cases, remaining_cases = train_test_split(
    eligible,
    test_size=0.4,
    random_state=42,
    stratify=eligible["source_category"],
)

validation_cases, test_cases = train_test_split(
    remaining_cases,
    test_size=0.5,
    random_state=42,
    stratify=remaining_cases["source_category"],
)

# 원본 사진이 양쪽에 섞이지 않았는지 확인
groups = [
    set(train_cases["filename"]),
    set(validation_cases["filename"]),
    set(test_cases["filename"]),
]
assert not (groups[0] & groups[1])
assert not (groups[0] & groups[2])
assert not (groups[1] & groups[2])

split_dir = BASE / "splits" / "tvg_number_v1"
split_dir.mkdir(parents=True, exist_ok=True)

for name, table in [
    ("train", train_cases),
    ("validation", validation_cases),
    ("test", test_cases),
]:
    table.to_csv(
        split_dir / f"{name}_cases.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print(f"{name}: 원본 사진 {len(table)}장")

excluded.to_csv(
    split_dir / "excluded_cases.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n제외한 사진:")
display(excluded[["filename", "source_category"]])
print("저장 위치:", split_dir)

In [ ]:
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
import pandas as pd
import re
from tqdm.auto import tqdm

number_names = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
number_to_id = {name: i for i, name in enumerate(number_names)}

output_root = BASE / "derived" / "number_shared_v1"


def extract_number_crops(case_table, split_name):
    records = []

    for _, case in tqdm(
        case_table.iterrows(),
        total=len(case_table),
        desc=f"{split_name} 조각 추출",
    ):
        with Image.open(BASE / case["image_path"]) as source:
            source_image = source.convert("RGB")

        root = ET.parse(BASE / case["annotation_path"]).getroot()
        prepared = []

        for index, obj in enumerate(root.findall("object")):
            label = (obj.findtext("name") or "").strip()

            if label in ("X", "Y"):
                chromosome = f"chr{label}"
            else:
                match = re.fullmatch(r"[A-G](\d+)", label)
                if not match:
                    raise ValueError(f"알 수 없는 라벨: {label}")
                chromosome = f"chr{int(match.group(1))}"

            original_box = tuple(
                int(obj.findtext(f"bndbox/{key}"))
                for key in ["xmin", "ymin", "xmax", "ymax"]
            )
            x1, y1, x2, y2 = original_box

            # 사진 밖으로 나간 좌표만 사진 경계로 제한
            box = (
                max(0, x1), max(0, y1),
                min(source_image.width, x2),
                min(source_image.height, y2),
            )
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError(
                    f"잘못된 상자: {case['filename']} / {index}"
                )

            prepared.append((
                index, chromosome, source_image.crop(box),
                box != original_box,
            ))

        # 원본 사진 하나 안에서는 공통 바탕 크기 사용
        side = max(max(crop.size) for _, _, crop, _ in prepared) + 12

        case_dir = output_root / split_name / Path(case["filename"]).stem
        case_dir.mkdir(parents=True, exist_ok=True)

        for index, chromosome, crop, clipped in prepared:
            canvas = Image.new("RGB", (side, side), "white")
            canvas.paste(
                crop,
                ((side - crop.width) // 2, (side - crop.height) // 2),
            )

            destination = case_dir / f"object_{index:03d}.png"
            canvas.save(destination)

            records.append({
                "source_image": case["filename"],
                "object_index": index,
                "chromosome": chromosome,
                "target": number_to_id[chromosome],
                "crop_path": str(destination),
                "bbox_clipped": clipped,
            })

    manifest = pd.DataFrame(records)
    manifest.to_csv(
        split_dir / f"{split_name}_crops.csv",
        index=False,
        encoding="utf-8-sig",
    )
    return manifest


train_crops = extract_number_crops(train_cases, "train")
validation_crops = extract_number_crops(validation_cases, "validation")

print("\n학습 조각:", len(train_crops), "개")
print("검증 조각:", len(validation_crops), "개")
print(
    "사진 경계로 좌표를 제한한 상자:",
    int(train_crops["bbox_clipped"].sum()
        + validation_crops["bbox_clipped"].sum()),
    "개",
)
print("저장 위치:", output_root)
print("조각 추출 완료 — 아직 학습 전")

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm

feature_dir = BASE / "features" / "number_shared_v1"
feature_dir.mkdir(parents=True, exist_ok=True)

def extract_tvg_features(table, split_name):
    table = table.reset_index(drop=True)
    batches = []

    tvg_number_encoder.eval()

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(table), 4),
            desc=f"{split_name} 특징 추출",
        ):
            images = []

            for path in table.iloc[start:start + 4]["crop_path"]:
                with Image.open(path) as source:
                    images.append(
                        tvg_transform(source.convert("RGB"))
                    )

            inputs = torch.stack(images).to(device)
            features = tvg_number_encoder(inputs)

            if not torch.isfinite(features).all().item():
                raise ValueError("특징에 NaN 또는 무한대가 있어.")

            batches.append(features.cpu())

    features = torch.cat(batches, dim=0)

    assert features.shape == (len(table), 1024)

    payload = {
        "features": features,
        "labels": torch.tensor(
            table["target"].to_numpy(), dtype=torch.long
        ),
        "source_images": table["source_image"].tolist(),
        "object_indices": table["object_index"].tolist(),
        "crop_paths": table["crop_path"].tolist(),
        "chromosome_names": number_names,
        "metadata": {
            "encoder_checkpoint": str(
                Path(r"C:\chromosome_ai")
                / "pretrained" / "CHROMA" / "CHROMA-num_ab.pth"
            ),
            "global_pool": "avg",
            "crop_method": "per_image_shared_canvas_plus_12",
            "preprocessing": str(tvg_transform),
            "split_directory": str(split_dir),
        },
    }

    destination = feature_dir / f"{split_name}_features.pt"
    torch.save(payload, destination)

    print(f"\n{split_name} 특징 크기:", tuple(features.shape))
    print("저장 위치:", destination)

    return payload

tvg_train_features = extract_tvg_features(train_crops, "train")
tvg_validation_features = extract_tvg_features(
    validation_crops, "validation"
)

print("\n특징 저장 완료 — 아직 새 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X_train = tvg_train_features["features"].numpy()
y_train = tvg_train_features["labels"].numpy()

X_val = tvg_validation_features["features"].numpy()
y_val = tvg_validation_features["labels"].numpy()

# 학습 자료에 24개 번호가 모두 있는지 확인
assert set(y_train.tolist()) == set(range(24))

tvg_adapted_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("새 번호 분류기 학습 시작")
tvg_adapted_classifier.fit(X_train, y_train)

train_predictions = tvg_adapted_classifier.predict(X_train)
val_predictions = tvg_adapted_classifier.predict(X_val)

metrics = pd.DataFrame([
    {
        "구분": name,
        "조각 수": len(actual),
        "맞힌 수": int((actual == predicted).sum()),
        "정확도": accuracy_score(actual, predicted),
        "Macro F1": f1_score(
            actual, predicted,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
    }
    for name, actual, predicted in [
        ("학습", y_train, train_predictions),
        ("검증", y_val, val_predictions),
    ]
]).set_index("구분")

display(metrics.round(4))

# 검증 자료에서 21번을 얼마나 잘 찾는지 확인
chr21_id = number_to_id["chr21"]
actual_21 = y_val == chr21_id
predicted_21 = val_predictions == chr21_id

tp = int((actual_21 & predicted_21).sum())
fp = int((~actual_21 & predicted_21).sum())
fn = int((actual_21 & ~predicted_21).sum())

print("\n검증 자료의 21번 분류 결과")
print("실제 21번:", int(actual_21.sum()), "개")
print("21번을 정확히 찾음:", tp, "개")
print("다른 번호를 21번으로 오분류:", fp, "개")
print("실제 21번을 놓침:", fn, "개")

# 기존 분류기와 별도 폴더에 저장
model_dir = BASE / "models" / "number_shared_linear_v1"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": tvg_adapted_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": tvg_train_features["metadata"],
    "metrics": metrics.to_dict(orient="index"),
}, model_dir / "classifier.joblib")

validation_results = validation_crops.copy()
validation_results["predicted_chromosome"] = [
    number_names[int(i)] for i in val_predictions
]
validation_results["correct"] = y_val == val_predictions

validation_results.to_csv(
    model_dir / "validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
metrics.to_csv(
    model_dir / "metrics.csv",
    encoding="utf-8-sig",
)

print("\n학습·저장 완료:", model_dir)
print("테스트 자료는 아직 평가하지 않았어.")

In [ ]:
records = []

for filename, group in validation_results.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    tp = int((actual & predicted).sum())
    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    records.append({
        "원본 사진": filename,
        "실제 21번 개수": int(actual.sum()),
        "예측 21번 개수": int(predicted.sum()),
        "정확히 찾음": tp,
        "다른 번호를 21번으로 오분류": fp,
        "21번 놓침": fn,
        "개수 일치": int(actual.sum()) == int(predicted.sum()),
        "21번 식별 모두 정확": fp == 0 and fn == 0,
    })

count_review = pd.DataFrame(records)

display(count_review)

print(
    "\n21번 개수가 일치한 사진:",
    f"{count_review['개수 일치'].sum()}/{len(count_review)}장"
)
print(
    "21번을 오분류·놓침 없이 구분한 사진:",
    f"{count_review['21번 식별 모두 정확'].sum()}/{len(count_review)}장"
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

candidate_classifiers = {1.0: tvg_adapted_classifier}
comparison_rows = []

for c_value in [0.001, 0.01, 0.1, 1.0]:
    print(f"C={c_value} 비교 중")

    if c_value not in candidate_classifiers:
        classifier = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=c_value,
                max_iter=3000,
                solver="lbfgs",
                random_state=42,
            ),
        )
        classifier.fit(X_train, y_train)
        candidate_classifiers[c_value] = classifier

    classifier = candidate_classifiers[c_value]
    train_pred = classifier.predict(X_train)
    val_pred = classifier.predict(X_val)

    actual_21 = y_val == number_to_id["chr21"]
    predicted_21 = val_pred == number_to_id["chr21"]

    per_image = pd.DataFrame({
        "source_image": validation_crops["source_image"].to_numpy(),
        "actual_21": actual_21.astype(int),
        "predicted_21": predicted_21.astype(int),
    }).groupby("source_image").sum()

    comparison_rows.append({
        "C": c_value,
        "학습 정확도": accuracy_score(y_train, train_pred),
        "검증 정확도": accuracy_score(y_val, val_pred),
        "검증 Macro F1": f1_score(
            y_val, val_pred,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
        "21번 개수 일치 사진": int(
            per_image["actual_21"].eq(
                per_image["predicted_21"]
            ).sum()
        ),
        "다른 번호를 21번으로 오분류": int(
            (~actual_21 & predicted_21).sum()
        ),
        "21번 놓침": int(
            (actual_21 & ~predicted_21).sum()
        ),
    })

regularization_comparison = pd.DataFrame(comparison_rows)
display(regularization_comparison.round(4))

print("비교 완료 — 기존 저장 모델은 변경하지 않았어.")

In [ ]:
# 검증 Macro F1이 가장 높은 설정 선택
selected_row = regularization_comparison.sort_values(
    ["검증 Macro F1", "C"],
    ascending=[False, True],
).iloc[0]

selected_c = float(selected_row["C"])
tvg_selected_classifier = candidate_classifiers[selected_c]

selected_model_dir = (
    BASE / "models" / "number_shared_linear_selected_v1"
)
selected_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": tvg_selected_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": tvg_train_features["metadata"],
    "selected_C": selected_c,
    "selection_metric": "validation_macro_f1",
    "validation_metrics": selected_row.to_dict(),
    "test_evaluated": False,
}, selected_model_dir / "classifier.joblib")

regularization_comparison.to_csv(
    selected_model_dir / "regularization_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("선택한 C:", selected_c)
print(f"검증 정확도: {selected_row['검증 정확도']:.2%}")
print(f"검증 Macro F1: {selected_row['검증 Macro F1']:.4f}")
print("저장 위치:", selected_model_dir)
print("모델 선택·저장 완료 — 테스트 평가 전")

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)

# 1. 기존과 동일한 방식으로 테스트 조각·특징 추출
test_crops = extract_number_crops(test_cases, "test")
tvg_test_features = extract_tvg_features(test_crops, "test")

X_test = tvg_test_features["features"].numpy()
y_test = tvg_test_features["labels"].numpy()

# 2. 저장한 모델을 불러와 평가
saved_bundle = joblib.load(
    selected_model_dir / "classifier.joblib"
)
test_classifier = saved_bundle["classifier"]

test_predictions = test_classifier.predict(X_test)

test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(
    y_test, test_predictions,
    labels=list(range(24)),
    average="macro",
    zero_division=0,
)

print("\n===== 테스트 번호 분류 결과 =====")
print("원본 사진:", len(test_cases), "장")
print("염색체 조각:", len(y_test), "개")
print(f"번호 정확도: {test_accuracy:.2%}")
print(f"Macro F1: {test_macro_f1:.4f}")

# 3. 조각별 예측 기록
test_results = test_crops.copy()
test_results["predicted_chromosome"] = [
    number_names[int(i)] for i in test_predictions
]
test_results["correct"] = y_test == test_predictions

# 4. 원본 사진별 21번 식별·개수 평가
records = []

for filename, group in test_results.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    tp = int((actual & predicted).sum())
    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    records.append({
        "원본 사진": filename,
        "실제 21번 개수": int(actual.sum()),
        "예측 21번 개수": int(predicted.sum()),
        "정확히 찾음": tp,
        "다른 번호를 21번으로 오분류": fp,
        "21번 놓침": fn,
        "개수 일치": int(actual.sum()) == int(predicted.sum()),
        "21번 식별 모두 정확": fp == 0 and fn == 0,
    })

test_count_review = pd.DataFrame(records)

print("\n===== 테스트 21번 평가 =====")
display(test_count_review)

print(
    "21번 개수가 일치한 사진:",
    f"{test_count_review['개수 일치'].sum()}/{len(test_count_review)}장",
)
print(
    "21번을 오분류·놓침 없이 구분한 사진:",
    f"{test_count_review['21번 식별 모두 정확'].sum()}"
    f"/{len(test_count_review)}장",
)

# 5. 번호별 성능과 결과 저장
report = classification_report(
    y_test, test_predictions,
    labels=list(range(24)),
    target_names=number_names,
    output_dict=True,
    zero_division=0,
)

evaluation_dir = selected_model_dir / "test_evaluation_v1"
evaluation_dir.mkdir(parents=True, exist_ok=True)

test_results.to_csv(
    evaluation_dir / "predictions.csv",
    index=False, encoding="utf-8-sig",
)
test_count_review.to_csv(
    evaluation_dir / "chr21_count_review.csv",
    index=False, encoding="utf-8-sig",
)
pd.DataFrame(report).T.to_csv(
    evaluation_dir / "classification_report.csv",
    encoding="utf-8-sig",
)
pd.DataFrame([{
    "source_images": len(test_cases),
    "chromosome_crops": len(y_test),
    "accuracy": test_accuracy,
    "macro_f1": test_macro_f1,
    "chr21_count_correct_images": int(
        test_count_review["개수 일치"].sum()
    ),
    "chr21_identification_correct_images": int(
        test_count_review["21번 식별 모두 정확"].sum()
    ),
}]).to_csv(
    evaluation_dir / "metrics.csv",
    index=False, encoding="utf-8-sig",
)

print("\n테스트 평가·결과 저장 완료:", evaluation_dir)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# 선택한 모델의 검증 예측을 다시 계산
selected_val_predictions = tvg_selected_classifier.predict(X_val)

validation_review = validation_crops.copy().reset_index(drop=True)
validation_review["predicted_chromosome"] = [
    number_names[int(i)] for i in selected_val_predictions
]
validation_review["correct"] = (
    validation_review["chromosome"]
    == validation_review["predicted_chromosome"]
)

# 여러 원본 사진의 오류를 볼 수 있도록 사진당 최대 2개 선택
errors = validation_review.loc[
    ~validation_review["correct"]
].copy()

examples = (
    errors.groupby("source_image", group_keys=False)
    .head(2)
    .sample(n=min(16, len(errors.groupby("source_image").head(2))),
            random_state=42)
)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for ax in axes.flat:
    ax.axis("off")

for ax, (_, record) in zip(axes.flat, examples.iterrows()):
    with Image.open(record["crop_path"]) as source:
        ax.imshow(source.convert("RGB"))

    ax.set_title(
        f"True: {record['chromosome']}\n"
        f"Pred: {record['predicted_chromosome']}\n"
        f"{record['source_image']}",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import xml.etree.ElementTree as ET

overlap_review = validation_review.copy()
overlap_review["max_box_overlap"] = np.nan

for _, case in validation_cases.iterrows():
    root = ET.parse(BASE / case["annotation_path"]).getroot()

    boxes = [
        tuple(
            int(obj.findtext(f"bndbox/{key}"))
            for key in ["xmin", "ymin", "xmax", "ymax"]
        )
        for obj in root.findall("object")
    ]

    for index, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        area = max(0, x2 - x1) * max(0, y2 - y1)
        if area == 0:
            raise ValueError(f"면적이 0인 상자: {case['filename']}")

        max_overlap = 0.0

        for other_index, other in enumerate(boxes):
            if index == other_index:
                continue

            ox1, oy1, ox2, oy2 = other
            intersection = (
                max(0, min(x2, ox2) - max(x1, ox1))
                * max(0, min(y2, oy2) - max(y1, oy1))
            )
            max_overlap = max(max_overlap, intersection / area)

        mask = (
            overlap_review["source_image"].eq(case["filename"])
            & overlap_review["object_index"].eq(index)
        )
        overlap_review.loc[mask, "max_box_overlap"] = max_overlap

assert overlap_review["max_box_overlap"].notna().all()

# 탐색용 기준: 자기 상자 면적의 20% 이상이 다른 상자와 겹침
overlap_review["상자 겹침 구분"] = np.where(
    overlap_review["max_box_overlap"] >= 0.2,
    "20% 이상",
    "20% 미만",
)

comparison = overlap_review.groupby("상자 겹침 구분").agg(
    조각_수=("correct", "size"),
    정답_수=("correct", "sum"),
    정확도=("correct", "mean"),
)

display(comparison.round(4))

In [ ]:
import copy
import numpy as np
import torch
from torch import nn
from PIL import Image

# 다시 실행할 때 이전 준비 모델의 GPU 메모리 해제
if "tvg_finetune_model" in globals():
    del tvg_finetune_model
    torch.cuda.empty_cache()

# 기존 인코더는 보존하고 복사본 사용
tvg_finetune_model = copy.deepcopy(tvg_number_encoder)

# 현재 분류기의 표준화 + 선형 분류를 하나의 분류층으로 변환
scaler = tvg_selected_classifier.named_steps["standardscaler"]
linear = tvg_selected_classifier.named_steps["logisticregression"]

assert np.array_equal(linear.classes_, np.arange(24))

weight = linear.coef_ / scaler.scale_[None, :]
bias = linear.intercept_ - weight @ scaler.mean_

tvg_finetune_model.head = nn.Linear(1024, 24).to(device)

with torch.no_grad():
    tvg_finetune_model.head.weight.copy_(
        torch.tensor(weight, dtype=torch.float32, device=device)
    )
    tvg_finetune_model.head.bias.copy_(
        torch.tensor(bias, dtype=torch.float32, device=device)
    )

# 전체 동결 후 마지막 블록·출력 정규화·분류층만 학습 허용
tvg_finetune_model.requires_grad_(False)

for module in [
    tvg_finetune_model.blocks[-1],
    tvg_finetune_model.fc_norm,
    tvg_finetune_model.head,
]:
    module.requires_grad_(True)

# 이전 분류기와 새 모델이 같은 예측으로 출발하는지 확인
images = []
for path in validation_crops.iloc[:4]["crop_path"]:
    with Image.open(path) as source:
        images.append(tvg_transform(source.convert("RGB")))

probe = torch.stack(images).to(device)

tvg_number_encoder.eval()
tvg_finetune_model.eval()

with torch.inference_mode():
    old_features = tvg_number_encoder(probe).cpu().numpy()
    old_predictions = tvg_selected_classifier.predict(old_features)
    new_predictions = (
        tvg_finetune_model(probe).argmax(dim=1).cpu().numpy()
    )

assert np.array_equal(old_predictions, new_predictions), (
    "출발 예측이 달라. 아직 학습하지 말고 이 오류를 알려줘."
)

trainable = sum(
    p.numel() for p in tvg_finetune_model.parameters()
    if p.requires_grad
)

finetune_dir = BASE / "models" / "number_shared_finetune_v1"
finetune_dir.mkdir(parents=True, exist_ok=True)

print("기존 분류기와 출발 예측 일치: True")
print(f"학습 가능한 파라미터: {trainable:,}개")
print("학습 범위: 마지막 인코더 블록 + fc_norm + 분류층")
print("저장 예정 위치:", finetune_dir)
print("미세조정 준비 완료 — 아직 학습 전")

In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


class TVGNumberDataset(Dataset):
    def __init__(self, table):
        self.table = table.reset_index(drop=True)

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        record = self.table.iloc[index]
        with Image.open(record["crop_path"]) as source:
            image = tvg_transform(source.convert("RGB"))
        return image, int(record["target"])


ft_train_loader = DataLoader(
    TVGNumberDataset(train_crops),
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

ft_val_loader = DataLoader(
    TVGNumberDataset(validation_crops),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        p for p in tvg_finetune_model.parameters()
        if p.requires_grad
    ],
    lr=1e-5,
    weight_decay=1e-4,
)


def evaluate_finetune():
    tvg_finetune_model.eval()
    actual, predicted = [], []
    loss_sum = 0.0

    with torch.inference_mode():
        for images, labels in tqdm(ft_val_loader, desc="검증"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = tvg_finetune_model(images)
            loss = criterion(logits, labels)

            loss_sum += loss.item() * len(labels)
            actual.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(1).cpu().tolist())

    return {
        "loss": loss_sum / len(actual),
        "accuracy": accuracy_score(actual, predicted),
        "macro_f1": f1_score(
            actual, predicted,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
    }


def save_finetune(epoch, metrics):
    # CPU 가중치로 저장해서 이후 다른 장치에서도 복원 가능
    torch.save({
        "model": {
            key: value.detach().cpu()
            for key, value in tvg_finetune_model.state_dict().items()
        },
        "epoch": epoch,
        "validation_metrics": metrics,
        "chromosome_names": number_names,
        "model_name": "vit_large_patch16_224",
        "global_pool": "avg",
        "num_classes": 24,
        "preprocessing": str(tvg_transform),
        "crop_method": "per_image_shared_canvas_plus_12",
        "split_directory": str(split_dir),
    }, finetune_dir / "best.pth")


# 출발 성능을 평가하고 먼저 저장
baseline = evaluate_finetune()
best_f1 = baseline["macro_f1"]
best_epoch = 0
no_improvement = 0

save_finetune(0, baseline)

history = [{
    "epoch": 0,
    "train_loss": None,
    "train_accuracy": None,
    "validation_loss": baseline["loss"],
    "validation_accuracy": baseline["accuracy"],
    "validation_macro_f1": baseline["macro_f1"],
}]

print(
    f"\n출발 성능 | 정확도 {baseline['accuracy']:.2%}"
    f" | Macro F1 {best_f1:.4f}"
)

for epoch in range(1, 6):
    # 고정한 앞부분은 평가 모드 유지
    tvg_finetune_model.eval()
    tvg_finetune_model.blocks[-1].train()
    tvg_finetune_model.fc_norm.train()
    tvg_finetune_model.head.train()

    loss_sum = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(
        ft_train_loader, desc=f"학습 {epoch}/5"
    ):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = tvg_finetune_model(images)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss).item():
            raise RuntimeError("Loss가 비정상이야. 학습을 중단했어.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in tvg_finetune_model.parameters()
             if p.requires_grad],
            max_norm=1.0,
        )
        optimizer.step()

        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    metrics = evaluate_finetune()

    improved = metrics["macro_f1"] > best_f1 + 1e-6
    if improved:
        best_f1 = metrics["macro_f1"]
        best_epoch = epoch
        no_improvement = 0
        save_finetune(epoch, metrics)
    else:
        no_improvement += 1

    history.append({
        "epoch": epoch,
        "train_loss": loss_sum / total,
        "train_accuracy": correct / total,
        "validation_loss": metrics["loss"],
        "validation_accuracy": metrics["accuracy"],
        "validation_macro_f1": metrics["macro_f1"],
    })

    pd.DataFrame(history).to_csv(
        finetune_dir / "history.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"\nEpoch {epoch}/5"
        f" | 학습 정확도 {correct / total:.2%}"
        f" | 검증 정확도 {metrics['accuracy']:.2%}"
        f" | 검증 Macro F1 {metrics['macro_f1']:.4f}"
        + (" ★ 최적 모델 저장" if improved else "")
    )

    if no_improvement >= 2:
        print("검증 Macro F1이 2에포크 연속 개선되지 않아 중단했어.")
        break

tvg_finetune_model.eval()

print("\n학습 완료")
print("최적 에포크:", best_epoch, "(0이면 출발 모델)")
print(f"최적 검증 Macro F1: {best_f1:.4f}")
print("저장 위치:", finetune_dir)
print("현재 메모리에는 마지막 에포크 모델이 있어.")
print("다음 평가는 저장된 best.pth를 복원해서 진행할 거야.")

In [ ]:
# 1. 가장 좋았던 모델 복원
checkpoint = torch.load(
    finetune_dir / "best.pth",
    map_location="cpu",
    weights_only=True,
    mmap=True,
)

tvg_finetune_model.load_state_dict(
    checkpoint["model"], strict=True
)
tvg_finetune_model.eval()

print("복원한 에포크:", checkpoint["epoch"])

# 2. 검증 조각 번호 예측
predictions = []

with torch.inference_mode():
    for images, _ in tqdm(ft_val_loader, desc="최적 모델 검증"):
        logits = tvg_finetune_model(images.to(device))
        predictions.extend(logits.argmax(1).cpu().tolist())

ft_validation_results = validation_crops.copy().reset_index(drop=True)
ft_validation_results["predicted_chromosome"] = [
    number_names[int(i)] for i in predictions
]

# 3. 사진별 21번 개수·식별 확인
records = []

for filename, group in ft_validation_results.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    records.append({
        "원본 사진": filename,
        "실제 21번 개수": int(actual.sum()),
        "예측 21번 개수": int(predicted.sum()),
        "다른 번호를 21번으로 오분류": fp,
        "21번 놓침": fn,
        "개수 일치": int(actual.sum()) == int(predicted.sum()),
        "21번 식별 모두 정확": fp == 0 and fn == 0,
    })

ft_count_review = pd.DataFrame(records)
display(ft_count_review)

print(
    "\n21번 개수 일치:",
    f"{ft_count_review['개수 일치'].sum()}/{len(ft_count_review)}장"
)
print(
    "21번 식별 모두 정확:",
    f"{ft_count_review['21번 식별 모두 정확'].sum()}"
    f"/{len(ft_count_review)}장"
)
print(
    "다른 번호를 21번으로 오분류:",
    ft_count_review["다른 번호를 21번으로 오분류"].sum(), "개"
)
print("21번 놓침:", ft_count_review["21번 놓침"].sum(), "개")

ft_validation_results.to_csv(
    finetune_dir / "validation_predictions.csv",
    index=False, encoding="utf-8-sig",
)
ft_count_review.to_csv(
    finetune_dir / "validation_chr21_count_review.csv",
    index=False, encoding="utf-8-sig",
)

In [ ]:
from pathlib import Path
import pandas as pd

normal_csv = BASE / "source_metadata" / "normal.csv"
normal_catalog = pd.read_csv(normal_csv)

print("정상 원본 목록:", len(normal_catalog), "행")
print("열 이름:", normal_catalog.columns.tolist())
display(normal_catalog.head())

# 열 이름 앞뒤 공백 제거
normal_catalog.columns = normal_catalog.columns.str.strip()

if "filename" not in normal_catalog.columns:
    raise ValueError("filename 열이 없어. 위 출력 결과를 보내줘.")

normal_names = (
    normal_catalog["filename"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
)

existing_case_names = set(catalog["filename"].astype(str))

print("\n중복 제거한 정상 파일명:", len(normal_names), "개")
print(
    "기존 사례 목록과 같은 파일명:",
    int(normal_names.isin(existing_case_names).sum()), "개"
)
print(
    "현재 images 폴더에 있는 정상 사진:",
    sum((BASE / "images" / name).is_file() for name in normal_names),
    "장"
)

In [ ]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import hashlib
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

download_base = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "aliabedimadiseh/chromosome-image-dataset-karyotype/"
)

# 같은 200장을 다시 선택할 수 있도록 고정
selected_normal_names = (
    normal_names.sort_values()
    .sample(n=200, random_state=42)
    .tolist()
)

# 선택 목록부터 저장
pd.DataFrame({"filename": selected_normal_names}).to_csv(
    BASE / "normal_selected_200_v1.csv",
    index=False,
    encoding="utf-8-sig",
)


def get_source_bytes(archive_path):
    url = download_base + urllib.parse.quote(archive_path, safe="")

    with urllib.request.urlopen(url, timeout=30) as response:
        data = response.read(10 * 1024 * 1024)

    if len(data) >= 10 * 1024 * 1024:
        raise ValueError("예상보다 큰 응답이야.")
    return data


def acquire_normal_case(filename):
    image_path = BASE / "images" / filename
    xml_name = Path(filename).with_suffix(".xml").name
    annotation_path = BASE / "annotations" / xml_name

    result = {
        "filename": filename,
        "source_category": "normal",
        "image_path": f"images/{filename}",
        "annotation_path": f"annotations/{xml_name}",
        "acquisition_status": "failed",
    }

    for attempt in range(2):
        try:
            image_bytes = (
                image_path.read_bytes()
                if image_path.is_file()
                else get_source_bytes(
                    f"Data/24_chromosomes_object/JEPG/{filename}"
                )
            )
            xml_bytes = (
                annotation_path.read_bytes()
                if annotation_path.is_file()
                else get_source_bytes(
                    f"Data/24_chromosomes_object/annotations/{xml_name}"
                )
            )

            # JPEG 실제 디코딩 확인
            with Image.open(BytesIO(image_bytes)) as source:
                source.load()
                image_size = source.size

            # XML 실제 파싱 및 사진 크기 확인
            root = ET.fromstring(xml_bytes)
            xml_size = (
                int(root.findtext("size/width")),
                int(root.findtext("size/height")),
            )
            if image_size != xml_size:
                raise ValueError("사진 크기와 XML 크기가 달라.")

            object_count = len(root.findall("object"))
            if object_count == 0:
                raise ValueError("번호 정답 상자가 없어.")

            # 두 파일 모두 확인한 후 저장
            for destination, data in [
                (image_path, image_bytes),
                (annotation_path, xml_bytes),
            ]:
                destination.parent.mkdir(parents=True, exist_ok=True)
                temporary = destination.with_suffix(
                    destination.suffix + ".part"
                )
                temporary.write_bytes(data)
                temporary.replace(destination)

            result.update({
                "acquisition_status": "paired",
                "xml_object_count": object_count,
                "image_sha256": hashlib.sha256(image_bytes).hexdigest(),
                "annotation_sha256": hashlib.sha256(xml_bytes).hexdigest(),
                "error": "",
            })
            return result

        except Exception as error:
            result["error"] = str(error)

    return result


results = []
manifest_path = BASE / "normal_acquisition_200_v1.csv"

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [
        pool.submit(acquire_normal_case, filename)
        for filename in selected_normal_names
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="정상 사진·주석 다운로드",
    ):
        results.append(future.result())

        # 진행 상황을 계속 저장
        pd.DataFrame(results).to_csv(
            manifest_path,
            index=False,
            encoding="utf-8-sig",
        )

normal_acquired = pd.DataFrame(results)
success = normal_acquired["acquisition_status"].eq("paired")

print("\n사진·주석 확보:", int(success.sum()), "쌍")
print("실패:", int((~success).sum()), "건")
print("목록 저장 위치:", manifest_path)

if success.any():
    print("\n사진별 정답 상자 수 분포:")
    display(
        normal_acquired.loc[success, "xml_object_count"]
        .value_counts()
        .sort_index()
        .rename_axis("정답 상자 수")
        .to_frame("사진 수")
    )

if (~success).any():
    print("\n실패 파일:")
    display(normal_acquired.loc[~success, ["filename", "error"]])

print("다운로드 단계 완료 — 아직 분할·조각 추출·학습 전")

In [ ]:
from collections import Counter
import xml.etree.ElementTree as ET
import re
import pandas as pd

audit_records = []

for _, case in normal_acquired.loc[success].iterrows():
    root = ET.parse(BASE / case["annotation_path"]).getroot()

    counts = Counter()
    unknown_labels = []
    invalid_boxes = 0
    outside_boxes = 0

    width = int(root.findtext("size/width"))
    height = int(root.findtext("size/height"))

    for obj in root.findall("object"):
        label = (obj.findtext("name") or "").strip()

        if label in ("X", "Y"):
            chromosome = f"chr{label}"
        else:
            match = re.fullmatch(r"[A-G](\d+)", label)
            chromosome = (
                f"chr{int(match.group(1))}" if match else None
            )

        if chromosome not in number_to_id:
            unknown_labels.append(label)
        else:
            counts[chromosome] += 1

        x1, y1, x2, y2 = [
            int(obj.findtext(f"bndbox/{key}"))
            for key in ["xmin", "ymin", "xmax", "ymax"]
        ]

        invalid_boxes += int(x2 <= x1 or y2 <= y1)
        outside_boxes += int(
            x1 < 0 or y1 < 0 or x2 > width or y2 > height
        )

    # 今回の学習に使う「標準的な46本構成」との一致を確認
    autosomes_ok = all(
        counts[f"chr{i}"] == 2 for i in range(1, 23)
    )
    sex_counts_ok = (
        counts["chrX"], counts["chrY"]
    ) in [(2, 0), (1, 1)]

    reasons = []
    if len(root.findall("object")) != 46:
        reasons.append("상자 총수 46개 아님")
    if not autosomes_ok:
        reasons.append("1~22번이 각각 2개인 구성과 다름")
    if not sex_counts_ok:
        reasons.append("XX 또는 XY 주석 구성과 다름")
    if unknown_labels:
        reasons.append("알 수 없는 번호 라벨")
    if invalid_boxes:
        reasons.append("면적이 잘못된 상자")
    if outside_boxes:
        reasons.append("사진 경계를 벗어난 상자")

    audit_records.append({
        "filename": case["filename"],
        "source_category": "normal",
        "image_path": case["image_path"],
        "annotation_path": case["annotation_path"],
        "xml_object_count": len(root.findall("object")),
        "annotation_check_pass": not reasons,
        "review_reason": " / ".join(reasons),
    })

normal_audit = pd.DataFrame(audit_records)

normal_eligible = normal_audit.loc[
    normal_audit["annotation_check_pass"]
].copy()

normal_needs_review = normal_audit.loc[
    ~normal_audit["annotation_check_pass"]
].copy()

normal_audit.to_csv(
    BASE / "normal_annotation_audit_v1.csv",
    index=False,
    encoding="utf-8-sig",
)

print("이번 주석 검사 통과:", len(normal_eligible), "장")
print("별도 검토 필요:", len(normal_needs_review), "장")

display(normal_needs_review[
    ["filename", "xml_object_count", "review_reason"]
])

print("주석 검사 완료 — 질환이나 정상 여부를 임상 판정한 것은 아니야.")

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# 완료 순서와 관계없이 같은 분할이 나오도록 정렬
normal_ready = (
    normal_eligible
    .sort_values("filename")
    .reset_index(drop=True)
    .copy()
)

normal_train_cases, normal_remaining = train_test_split(
    normal_ready,
    test_size=0.4,
    random_state=42,
)

normal_validation_cases, normal_test_cases = train_test_split(
    normal_remaining,
    test_size=0.5,
    random_state=42,
)

# 기존 자료와 추가 자료를 구분하는 출처 표시
def combine_cases(original, additional):
    original = original.copy()
    additional = additional.copy()

    original["data_cohort"] = "original_abnormal_cases"
    additional["data_cohort"] = "additional_normal_cases"

    return pd.concat(
        [original, additional],
        ignore_index=True,
        sort=False,
    )

expanded_train_cases = combine_cases(
    train_cases, normal_train_cases
)
expanded_validation_cases = combine_cases(
    validation_cases, normal_validation_cases
)
expanded_test_cases = combine_cases(
    test_cases, normal_test_cases
)

groups = [
    set(expanded_train_cases["filename"]),
    set(expanded_validation_cases["filename"]),
    set(expanded_test_cases["filename"]),
]

assert not (groups[0] & groups[1])
assert not (groups[0] & groups[2])
assert not (groups[1] & groups[2])

# 중복 파일명 및 기존 제외 사례 유입 확인
for table in [
    expanded_train_cases,
    expanded_validation_cases,
    expanded_test_cases,
]:
    assert table["filename"].is_unique
    assert not table["filename"].isin(
        ["1052521.jpg", "1080904.jpg"]
    ).any()

expanded_split_dir = BASE / "splits" / "tvg_number_expanded_v1"
expanded_split_dir.mkdir(parents=True, exist_ok=True)

for name, table in [
    ("train", expanded_train_cases),
    ("validation", expanded_validation_cases),
    ("test", expanded_test_cases),
]:
    table.to_csv(
        expanded_split_dir / f"{name}_cases.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print(f"{name}: 원본 사진 {len(table)}장")

print("\n분할 간 같은 파일명 없음: True")
print("저장 위치:", expanded_split_dir)
print("확장 분할 완료 — 추가 조각 추출·학습 전")

In [ ]:
# 이번 확장 실험의 저장 경로
# 앞서 만든 추출 함수가 사용하는 경로를 변경
split_dir = expanded_split_dir
output_root = BASE / "derived" / "number_shared_expanded_v1"

# 새로 추가한 사진만 조각 추출
additional_train_crops = extract_number_crops(
    normal_train_cases, "train_additional"
)
additional_validation_crops = extract_number_crops(
    normal_validation_cases, "validation_additional"
)


def combine_crop_manifests(original, additional, case_table):
    combined = pd.concat(
        [original, additional],
        ignore_index=True,
    )

    # 원본 사진의 출처 연결
    combined = combined.drop(columns=["data_cohort"], errors="ignore")
    combined = combined.merge(
        case_table[["filename", "data_cohort"]],
        left_on="source_image",
        right_on="filename",
        how="left",
        validate="many_to_one",
    ).drop(columns=["filename"])

    assert combined["data_cohort"].notna().all()
    assert not combined.duplicated(
        ["source_image", "object_index"]
    ).any()

    return combined


expanded_train_crops = combine_crop_manifests(
    train_crops,
    additional_train_crops,
    expanded_train_cases,
)

expanded_validation_crops = combine_crop_manifests(
    validation_crops,
    additional_validation_crops,
    expanded_validation_cases,
)

# 기존 실험과 별도 폴더에 전체 목록 저장
for name, table in [
    ("train", expanded_train_crops),
    ("validation", expanded_validation_crops),
]:
    table.to_csv(
        expanded_split_dir / f"{name}_crops.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"{name}: 원본 {table['source_image'].nunique()}장"
        f" / 조각 {len(table):,}개"
    )

print(
    "\n추가 자료에서 사진 경계로 좌표를 제한한 상자:",
    int(
        additional_train_crops["bbox_clipped"].sum()
        + additional_validation_crops["bbox_clipped"].sum()
    ),
    "개",
)
print("조각 목록 저장 위치:", expanded_split_dir)
print("확장 조각 추출 완료 — 아직 추가 특징 추출·학습 전")

In [ ]:
import torch

# 확장 실험용 특징 저장 폴더
feature_dir = BASE / "features" / "number_shared_expanded_v1"
feature_dir.mkdir(parents=True, exist_ok=True)

# 기존 함수는 미세조정 전 tvg_number_encoder를 사용
additional_train_features = extract_tvg_features(
    additional_train_crops, "train_additional"
)
additional_validation_features = extract_tvg_features(
    additional_validation_crops, "validation_additional"
)


def combine_feature_payloads(original, additional, crop_table, split_name):
    assert original["chromosome_names"] == additional["chromosome_names"]
    assert (
        original["metadata"]["encoder_checkpoint"]
        == additional["metadata"]["encoder_checkpoint"]
    )
    assert (
        original["metadata"]["preprocessing"]
        == additional["metadata"]["preprocessing"]
    )

    combined = {
        "features": torch.cat(
            [original["features"], additional["features"]], dim=0
        ),
        "labels": torch.cat(
            [original["labels"], additional["labels"]], dim=0
        ),
        "chromosome_names": original["chromosome_names"],
        "metadata": {
            **additional["metadata"],
            "split_directory": str(expanded_split_dir),
            "experiment": "normal_data_expansion_v1",
        },
    }

    for key in ["source_images", "object_indices", "crop_paths"]:
        combined[key] = original[key] + additional[key]

    # 特徴と正解・写真一覧の並びが一致するか確認
    assert combined["crop_paths"] == crop_table["crop_path"].tolist()
    assert combined["source_images"] == crop_table["source_image"].tolist()
    assert combined["object_indices"] == crop_table["object_index"].tolist()
    assert torch.equal(
        combined["labels"],
        torch.tensor(crop_table["target"].to_numpy(), dtype=torch.long),
    )
    assert combined["features"].shape == (len(crop_table), 1024)
    assert torch.isfinite(combined["features"]).all().item()

    combined["data_cohorts"] = crop_table["data_cohort"].tolist()

    destination = feature_dir / f"{split_name}_features.pt"
    torch.save(combined, destination)

    print(f"\n{split_name} 統合特徴:", tuple(combined["features"].shape))
    print("保存先:", destination)
    return combined


expanded_train_features = combine_feature_payloads(
    tvg_train_features,
    additional_train_features,
    expanded_train_crops,
    "train",
)

expanded_validation_features = combine_feature_payloads(
    tvg_validation_features,
    additional_validation_features,
    expanded_validation_crops,
    "validation",
)

print("\n확장 특징 저장 완료 — 아직 새 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

expanded_X_train = expanded_train_features["features"].numpy()
expanded_y_train = expanded_train_features["labels"].numpy()

expanded_X_val = expanded_validation_features["features"].numpy()
expanded_y_val = expanded_validation_features["labels"].numpy()

expanded_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.01,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("확장 데이터 번호 분류기 학습 시작")
expanded_classifier.fit(expanded_X_train, expanded_y_train)

expanded_train_pred = expanded_classifier.predict(expanded_X_train)
expanded_val_pred = expanded_classifier.predict(expanded_X_val)

cohorts = np.array(
    expanded_validation_features["data_cohorts"]
)

def summarize_numbers(name, actual, predicted):
    return {
        "평가 자료": name,
        "조각 수": len(actual),
        "맞힌 수": int((actual == predicted).sum()),
        "정확도": accuracy_score(actual, predicted),
        "Macro F1": f1_score(
            actual, predicted,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
    }

rows = [
    summarize_numbers(
        "학습 전체",
        expanded_y_train,
        expanded_train_pred,
    ),
    summarize_numbers(
        "검증 전체 57장",
        expanded_y_val,
        expanded_val_pred,
    ),
]

for cohort, title in [
    ("original_abnormal_cases", "기존 검증 20장"),
    ("additional_normal_cases", "추가 검증 37장"),
]:
    mask = cohorts == cohort
    rows.append(summarize_numbers(
        title,
        expanded_y_val[mask],
        expanded_val_pred[mask],
    ))

expanded_metrics = pd.DataFrame(rows).set_index("평가 자료")
display(expanded_metrics.round(4))

# 기존 검증 20장에서 21번 개수 판단도 비교
expanded_validation_results = (
    expanded_validation_crops.copy().reset_index(drop=True)
)
expanded_validation_results["predicted_chromosome"] = [
    number_names[int(i)] for i in expanded_val_pred
]

count_records = []

original_validation = expanded_validation_results.loc[
    expanded_validation_results["data_cohort"]
    == "original_abnormal_cases"
]

for filename, group in original_validation.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    count_records.append({
        "원본 사진": filename,
        "실제 21번 개수": int(actual.sum()),
        "예측 21번 개수": int(predicted.sum()),
        "오분류": fp,
        "놓침": fn,
        "개수 일치": int(actual.sum()) == int(predicted.sum()),
        "21번 식별 모두 정확": fp == 0 and fn == 0,
    })

expanded_count_review = pd.DataFrame(count_records)

print("\n기존 검증 20장의 21번 평가")
print(
    "개수 일치:",
    f"{expanded_count_review['개수 일치'].sum()}/20장",
)
print(
    "오분류·놓침 없이 식별:",
    f"{expanded_count_review['21번 식별 모두 정확'].sum()}/20장",
)
print("다른 번호를 21번으로 오분류:",
      expanded_count_review["오분류"].sum(), "개")
print("21번 놓침:",
      expanded_count_review["놓침"].sum(), "개")

# 별도 폴더에 모델과 평가 기록 저장
expanded_model_dir = (
    BASE / "models" / "number_shared_expanded_linear_v1"
)
expanded_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": expanded_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": expanded_train_features["metadata"],
    "C": 0.01,
    "metrics": expanded_metrics.to_dict(orient="index"),
}, expanded_model_dir / "classifier.joblib")

expanded_metrics.to_csv(
    expanded_model_dir / "metrics.csv",
    encoding="utf-8-sig",
)
expanded_validation_results.to_csv(
    expanded_model_dir / "validation_predictions.csv",
    index=False, encoding="utf-8-sig",
)
expanded_count_review.to_csv(
    expanded_model_dir / "original_validation_chr21_review.csv",
    index=False, encoding="utf-8-sig",
)

print("\n학습·저장 완료:", expanded_model_dir)

In [ ]:
comparison_rows = []

models_to_compare = {
    "기존 모델": tvg_selected_classifier,
    "확장 모델": expanded_classifier,
}

evaluation_groups = {
    "전체 57장": np.ones(len(expanded_y_val), dtype=bool),
    "기존 20장": cohorts == "original_abnormal_cases",
    "추가 37장": cohorts == "additional_normal_cases",
}

for model_name, classifier in models_to_compare.items():
    predictions = classifier.predict(expanded_X_val)

    for group_name, mask in evaluation_groups.items():
        actual = expanded_y_val[mask]
        predicted = predictions[mask]

        actual_21 = actual == number_to_id["chr21"]
        predicted_21 = predicted == number_to_id["chr21"]

        photo_review = pd.DataFrame({
            "source_image": expanded_validation_crops.loc[
                mask, "source_image"
            ].to_numpy(),
            "actual_21": actual_21.astype(int),
            "predicted_21": predicted_21.astype(int),
            "fp": (~actual_21 & predicted_21).astype(int),
            "fn": (actual_21 & ~predicted_21).astype(int),
        }).groupby("source_image").sum()

        comparison_rows.append({
            "모델": model_name,
            "검증 자료": group_name,
            "번호 정확도": accuracy_score(actual, predicted),
            "Macro F1": f1_score(
                actual, predicted,
                labels=list(range(24)),
                average="macro",
                zero_division=0,
            ),
            "21번 개수 일치 사진": int(
                photo_review["actual_21"].eq(
                    photo_review["predicted_21"]
                ).sum()
            ),
            "21번 식별 모두 정확한 사진": int(
                (
                    photo_review["fp"].eq(0)
                    & photo_review["fn"].eq(0)
                ).sum()
            ),
            "21번 오분류": int(photo_review["fp"].sum()),
            "21번 놓침": int(photo_review["fn"].sum()),
        })

fair_comparison = pd.DataFrame(comparison_rows)
display(fair_comparison.round(4))

fair_comparison.to_csv(
    expanded_model_dir / "same_validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("동일한 검증 자료 비교 완료 — 추가 학습 없음")

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm

rotation_feature_dir = BASE / "features" / "number_rotation_v1"
rotation_feature_dir.mkdir(parents=True, exist_ok=True)

rotation_operations = {
    90: Image.Transpose.ROTATE_90,
    180: Image.Transpose.ROTATE_180,
    270: Image.Transpose.ROTATE_270,
}

rotation_feature_paths = {}

tvg_number_encoder.eval()

for angle, operation in rotation_operations.items():
    destination = rotation_feature_dir / f"train_rot{angle}_features.pt"

    # 완료된 파일이 있으면 내용 확인 후 재사용
    if destination.is_file():
        cached = torch.load(
            destination,
            map_location="cpu",
            weights_only=True,
        )
        assert cached["angle"] == angle
        assert cached["crop_paths"] == expanded_train_crops["crop_path"].tolist()
        assert cached["features"].shape == (len(expanded_train_crops), 1024)
        assert torch.equal(
            cached["labels"], expanded_train_features["labels"]
        )
        assert cached["metadata"] == expanded_train_features["metadata"]
        assert torch.isfinite(cached["features"]).all().item()

        rotation_feature_paths[angle] = destination
        print(f"{angle}도: 저장된 특징 재사용")
        del cached
        continue

    batches = []

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(expanded_train_crops), 4),
            desc=f"학습 {angle}도 특징 추출",
        ):
            inputs = []

            for path in expanded_train_crops.iloc[
                start:start + 4
            ]["crop_path"]:
                with Image.open(path) as source:
                    rotated = source.convert("RGB").transpose(operation)
                    inputs.append(tvg_transform(rotated))

            batch = torch.stack(inputs).to(device)
            features = tvg_number_encoder(batch).cpu()

            if not torch.isfinite(features).all().item():
                raise ValueError("특징에 비정상 값이 있어.")

            batches.append(features)

    features = torch.cat(batches, dim=0)
    assert features.shape == (len(expanded_train_crops), 1024)

    payload = {
        "features": features,
        "labels": expanded_train_features["labels"],
        "crop_paths": expanded_train_crops["crop_path"].tolist(),
        "angle": angle,
        "metadata": expanded_train_features["metadata"],
    }

    temporary = destination.with_suffix(".pt.part")
    torch.save(payload, temporary)
    temporary.replace(destination)

    rotation_feature_paths[angle] = destination

    print(f"{angle}도 특징 크기:", tuple(features.shape))
    del batches, features, payload

print("\n회전 특징 추출 완료")
print("원본 0도 + 회전 3종 = 학습 입력 31,464개")
print("원본 학습 사진 수는 그대로 171장이야.")
print("검증 사진에는 회전을 추가하지 않았어.")
print("아직 회전 자료를 이용한 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# 1. 원본 + 회전 특징 합치기
feature_parts = [expanded_train_features["features"].numpy()]
label_parts = [expanded_train_features["labels"].numpy()]

for angle in [90, 180, 270]:
    payload = torch.load(
        rotation_feature_paths[angle],
        map_location="cpu",
        weights_only=True,
    )

    assert payload["crop_paths"] == expanded_train_crops["crop_path"].tolist()
    assert torch.equal(
        payload["labels"], expanded_train_features["labels"]
    )
    assert payload["metadata"] == expanded_train_features["metadata"]

    feature_parts.append(payload["features"].numpy())
    label_parts.append(payload["labels"].numpy())

rotation_X_train = np.concatenate(feature_parts, axis=0)
rotation_y_train = np.concatenate(label_parts, axis=0)

del feature_parts, label_parts, payload

# 2. 분류기 학습
rotation_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.01,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("학습 입력:", rotation_X_train.shape)
print("회전 증강 분류기 학습 시작 — 진행 막대 없이 실행돼.")

rotation_classifier.fit(
    rotation_X_train,
    rotation_y_train,
    logisticregression__sample_weight=np.full(
        len(rotation_y_train), 0.25
    ),
)

print("학습 완료. 동일 검증 자료 비교 중")

# 3. 회전 없는 검증 자료로 두 모델 비교
cohort_array = np.array(
    expanded_validation_features["data_cohorts"]
)
groups = {
    "전체 57장": np.ones(len(expanded_y_val), dtype=bool),
    "기존 20장": cohort_array == "original_abnormal_cases",
    "추가 37장": cohort_array == "additional_normal_cases",
}

rows = []

for model_name, classifier in [
    ("회전 추가 전", expanded_classifier),
    ("회전 추가 후", rotation_classifier),
]:
    predictions = classifier.predict(expanded_X_val)

    for group_name, mask in groups.items():
        actual = expanded_y_val[mask]
        predicted = predictions[mask]

        actual_21 = actual == number_to_id["chr21"]
        predicted_21 = predicted == number_to_id["chr21"]

        photo_counts = pd.DataFrame({
            "source_image": expanded_validation_crops.loc[
                mask, "source_image"
            ].to_numpy(),
            "actual": actual_21.astype(int),
            "predicted": predicted_21.astype(int),
            "fp": (~actual_21 & predicted_21).astype(int),
            "fn": (actual_21 & ~predicted_21).astype(int),
        }).groupby("source_image").sum()

        rows.append({
            "모델": model_name,
            "검증 자료": group_name,
            "번호 정확도": accuracy_score(actual, predicted),
            "Macro F1": f1_score(
                actual, predicted,
                labels=list(range(24)),
                average="macro",
                zero_division=0,
            ),
            "21번 개수 일치 사진": int(
                photo_counts["actual"].eq(
                    photo_counts["predicted"]
                ).sum()
            ),
            "21번 식별 모두 정확한 사진": int(
                (
                    photo_counts["fp"].eq(0)
                    & photo_counts["fn"].eq(0)
                ).sum()
            ),
            "21번 오분류": int(photo_counts["fp"].sum()),
            "21번 놓침": int(photo_counts["fn"].sum()),
        })

rotation_comparison = pd.DataFrame(rows)
display(rotation_comparison.round(4))

# 4. 기존 모델과 별도로 저장
rotation_model_dir = BASE / "models" / "number_rotation_linear_v1"
rotation_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": rotation_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": expanded_train_features["metadata"],
    "C": 0.01,
    "training_rotations": [0, 90, 180, 270],
    "sample_weight_per_rotation": 0.25,
    "validation_preprocessing": "original_orientation",
}, rotation_model_dir / "classifier.joblib")

rotation_comparison.to_csv(
    rotation_model_dir / "validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n학습·비교·저장 완료:", rotation_model_dir)

In [ ]:
import argparse
import numpy as np
import torch
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import accuracy_score, f1_score

# 1. 원본 체크포인트의 분류층만 읽기
checkpoint_path = (
    Path(r"C:\chromosome_ai")
    / "pretrained" / "CHROMA" / "CHROMA-num_ab.pth"
)

with torch.serialization.safe_globals([argparse.Namespace]):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
        mmap=True,
    )

head_weight = checkpoint["model"]["head.weight"].numpy().copy()
head_bias = checkpoint["model"]["head.bias"].numpy().copy()

assert head_weight.shape == (24, 1024)
assert head_bias.shape == (24,)

del checkpoint

# 2. 学習資料だけで出力インデックスと番号の対応を推定
raw_train_predictions = (
    expanded_X_train @ head_weight.T + head_bias
).argmax(axis=1)

matching_counts = np.zeros((24, 24), dtype=np.int64)

np.add.at(
    matching_counts,
    (expanded_y_train, raw_train_predictions),
    1,
)

# 各出力インデックスを異なる番号へ一対一で対応させる
true_ids, output_ids = linear_sum_assignment(-matching_counts)

inferred_mapping = np.empty(24, dtype=np.int64)
inferred_mapping[output_ids] = true_ids

mapped_train_predictions = inferred_mapping[raw_train_predictions]

print(
    "対応推定後の学習正解率:",
    f"{accuracy_score(expanded_y_train, mapped_train_predictions):.2%}",
)
print(
    "学習資料で出現した出力クラス:",
    len(np.unique(raw_train_predictions)),
    "/24",
)

# 3. 推定した対応を固定して検証資料へ適用
raw_val_predictions = (
    expanded_X_val @ head_weight.T + head_bias
).argmax(axis=1)

head_val_predictions = inferred_mapping[raw_val_predictions]

rows = []

for group_name, mask in evaluation_groups.items():
    actual = expanded_y_val[mask]
    predicted = head_val_predictions[mask]

    rows.append({
        "검증 자료": group_name,
        "번호 정확도": accuracy_score(actual, predicted),
        "Macro F1": f1_score(
            actual, predicted,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
    })

head_diagnostic = pd.DataFrame(rows)
display(head_diagnostic.round(4))

print("기존 분류층 진단 완료 — 가중치 학습 없음")
print("번호 대응은 학습 자료로 추정했으며 공식 대응표가 아니야.")

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

# 공식 build_chr_transform의 w_pad=False 설정을 참고
official_style_transform = transforms.Compose([
    transforms.Resize(
        224,
        interpolation=transforms.InterpolationMode.BICUBIC,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

preprocess_feature_dir = (
    BASE / "features" / "number_bicubic_rgb_v1"
)
preprocess_feature_dir.mkdir(parents=True, exist_ok=True)


def extract_preprocess_features(table, split_name):
    table = table.reset_index(drop=True)
    destination = preprocess_feature_dir / f"{split_name}_features.pt"

    metadata = {
        **expanded_train_features["metadata"],
        "preprocessing": str(official_style_transform),
        "experiment": "bicubic_rgb_preprocessing_v1",
    }

    # 완료된 파일은 내용 확인 후 재사용
    if destination.is_file():
        saved = torch.load(
            destination,
            map_location="cpu",
            weights_only=True,
        )
        assert saved["metadata"] == metadata
        assert saved["crop_paths"] == table["crop_path"].tolist()
        assert saved["features"].shape == (len(table), 1024)
        assert torch.equal(
            saved["labels"],
            torch.tensor(table["target"].to_numpy(), dtype=torch.long),
        )
        assert torch.isfinite(saved["features"]).all().item()

        print(f"{split_name}: 저장된 특징 재사용")
        return saved

    batches = []
    tvg_number_encoder.eval()

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(table), 4),
            desc=f"{split_name} 새 전처리 특징",
        ):
            images = []

            for path in table.iloc[start:start + 4]["crop_path"]:
                with Image.open(path) as source:
                    image = source.convert("RGB")

                    # 현재 공통 바탕 조각은 정사각형이어야 함
                    assert image.width == image.height
                    images.append(official_style_transform(image))

            inputs = torch.stack(images).to(device)
            features = tvg_number_encoder(inputs).cpu()

            if not torch.isfinite(features).all().item():
                raise ValueError("특징에 비정상 값이 있어.")

            batches.append(features)

    features = torch.cat(batches, dim=0)
    assert features.shape == (len(table), 1024)

    payload = {
        "features": features,
        "labels": torch.tensor(
            table["target"].to_numpy(), dtype=torch.long
        ),
        "crop_paths": table["crop_path"].tolist(),
        "source_images": table["source_image"].tolist(),
        "object_indices": table["object_index"].tolist(),
        "data_cohorts": table["data_cohort"].tolist(),
        "chromosome_names": number_names,
        "metadata": metadata,
    }

    temporary = destination.with_suffix(".pt.part")
    torch.save(payload, temporary)
    temporary.replace(destination)

    print(f"\n{split_name} 특징 크기:", tuple(features.shape))
    print("저장 위치:", destination)
    return payload


preprocess_train_features = extract_preprocess_features(
    expanded_train_crops, "train"
)
preprocess_validation_features = extract_preprocess_features(
    expanded_validation_crops, "validation"
)

print("\n새 전처리 특징 저장 완료 — 아직 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

new_X_train = preprocess_train_features["features"].numpy()
new_y_train = preprocess_train_features["labels"].numpy()
new_X_val = preprocess_validation_features["features"].numpy()
new_y_val = preprocess_validation_features["labels"].numpy()

# 두 전처리의 평가 대상·순서가 같은지 확인
assert (
    preprocess_validation_features["crop_paths"]
    == expanded_validation_features["crop_paths"]
)
assert np.array_equal(new_y_val, expanded_y_val)

preprocess_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.01,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("새 전처리 분류기 학습 시작")
preprocess_classifier.fit(new_X_train, new_y_train)

cohort_array = np.array(
    preprocess_validation_features["data_cohorts"]
)

groups = {
    "전체 57장": np.ones(len(new_y_val), dtype=bool),
    "기존 20장": cohort_array == "original_abnormal_cases",
    "추가 37장": cohort_array == "additional_normal_cases",
}

rows = []
new_val_predictions = None

for model_name, classifier, features in [
    ("기존 전처리", expanded_classifier, expanded_X_val),
    ("새 전처리", preprocess_classifier, new_X_val),
]:
    predictions = classifier.predict(features)

    if model_name == "새 전처리":
        new_val_predictions = predictions

    for group_name, mask in groups.items():
        actual = new_y_val[mask]
        predicted = predictions[mask]

        actual_21 = actual == number_to_id["chr21"]
        predicted_21 = predicted == number_to_id["chr21"]

        photo_counts = pd.DataFrame({
            "source_image": np.array(
                preprocess_validation_features["source_images"]
            )[mask],
            "actual": actual_21.astype(int),
            "predicted": predicted_21.astype(int),
            "fp": (~actual_21 & predicted_21).astype(int),
            "fn": (actual_21 & ~predicted_21).astype(int),
        }).groupby("source_image").sum()

        rows.append({
            "전처리": model_name,
            "검증 자료": group_name,
            "번호 정확도": accuracy_score(actual, predicted),
            "Macro F1": f1_score(
                actual, predicted,
                labels=list(range(24)),
                average="macro",
                zero_division=0,
            ),
            "21번 개수 일치 사진": int(
                photo_counts["actual"].eq(
                    photo_counts["predicted"]
                ).sum()
            ),
            "21번 식별 모두 정확한 사진": int(
                (
                    photo_counts["fp"].eq(0)
                    & photo_counts["fn"].eq(0)
                ).sum()
            ),
            "21번 오분류": int(photo_counts["fp"].sum()),
            "21번 놓침": int(photo_counts["fn"].sum()),
        })

preprocess_comparison = pd.DataFrame(rows)
display(preprocess_comparison.round(4))

# 새 전처리 모델은 별도로 저장
preprocess_model_dir = BASE / "models" / "number_bicubic_rgb_linear_v1"
preprocess_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": preprocess_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": preprocess_train_features["metadata"],
    "C": 0.01,
}, preprocess_model_dir / "classifier.joblib")

preprocess_comparison.to_csv(
    preprocess_model_dir / "validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

new_validation_results = expanded_validation_crops.copy()
new_validation_results["predicted_chromosome"] = [
    number_names[int(i)] for i in new_val_predictions
]
new_validation_results.to_csv(
    preprocess_model_dir / "validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n학습·비교·저장 완료:", preprocess_model_dir)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from scipy import ndimage

# 여러 원본 사진에서 조각 하나씩 선택
demo_crops = (
    expanded_train_crops
    .sort_values(["source_image", "object_index"])
    .drop_duplicates("source_image")
    .sample(n=6, random_state=42)
)


def make_trial_mask(rgb):
    gray = np.asarray(
        Image.fromarray(rgb).convert("L"),
        dtype=np.float32,
    )

    # 임계값 계산에 사용할 영상의 작은 잡음 완화
    smoothed = ndimage.gaussian_filter(gray, sigma=0.6)
    values = np.clip(smoothed, 0, 255).astype(np.uint8)

    # Otsu 방식으로 밝은 배경과 어두운 영역 구분
    histogram = np.bincount(
        values.ravel(), minlength=256
    ).astype(float)

    probability = histogram / histogram.sum()
    cumulative_weight = np.cumsum(probability)
    cumulative_mean = np.cumsum(
        probability * np.arange(256)
    )
    total_mean = cumulative_mean[-1]

    denominator = cumulative_weight * (1 - cumulative_weight)
    score = np.full(256, -1.0)

    valid = denominator > 0
    score[valid] = (
        (total_mean * cumulative_weight[valid]
         - cumulative_mean[valid]) ** 2
        / denominator[valid]
    )

    threshold = int(np.argmax(score))
    foreground = values <= threshold

    # 연결된 어두운 영역 중 가장 큰 덩어리 선택
    components, count = ndimage.label(
        foreground,
        structure=np.ones((3, 3), dtype=int),
    )

    if count == 0:
        return np.zeros_like(foreground)

    sizes = np.bincount(components.ravel())
    sizes[0] = 0
    return components == sizes.argmax()


fig, axes = plt.subplots(6, 3, figsize=(10, 18))

for row_index, (_, record) in enumerate(demo_crops.iterrows()):
    with Image.open(record["crop_path"]) as source:
        rgb = np.asarray(source.convert("RGB"))

    mask = make_trial_mask(rgb)

    cleaned = rgb.copy()
    cleaned[~mask] = 255

    axes[row_index, 0].imshow(rgb)
    axes[row_index, 1].imshow(
        mask, cmap="gray", vmin=0, vmax=1
    )
    axes[row_index, 2].imshow(cleaned)

    axes[row_index, 0].set_title(
        f"Original: {record['chromosome']}\n"
        f"{record['source_image']}",
        fontsize=9,
    )
    axes[row_index, 1].set_title("Trial mask")
    axes[row_index, 2].set_title("Masked image")

    for ax in axes[row_index]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("분리 가능성 확인용 결과 — 정답 마스크가 아니야.")

In [ ]:
# 서로 다른 번호에서 하나씩 선택
selected_examples = []

for chromosome in ["chr1", "chr5", "chr10", "chr15", "chr21", "chrY"]:
    candidates = expanded_train_crops.loc[
        expanded_train_crops["chromosome"].eq(chromosome)
    ].sort_values(["source_image", "object_index"])

    selected_examples.append(
        candidates.sample(n=1, random_state=42).iloc[0]
    )

fig, axes = plt.subplots(6, 3, figsize=(10, 18))

for row_index, record in enumerate(selected_examples):
    with Image.open(record["crop_path"]) as source:
        rgb = np.asarray(source.convert("RGB"))

    mask = make_trial_mask(rgb)
    cleaned = rgb.copy()
    cleaned[~mask] = 255

    axes[row_index, 0].imshow(rgb)
    axes[row_index, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[row_index, 2].imshow(cleaned)

    axes[row_index, 0].set_title(
        f"Original: {record['chromosome']}\n"
        f"{record['source_image']}",
        fontsize=9,
    )
    axes[row_index, 1].set_title("Trial mask")
    axes[row_index, 2].set_title("Masked image")

    for ax in axes[row_index]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

mask_output_root = BASE / "derived" / "number_background_mask_v1"
mask_manifest_dir = BASE / "splits" / "number_background_mask_v1"
mask_manifest_dir.mkdir(parents=True, exist_ok=True)


def save_masked_crops(table, split_name):
    result = table.copy().reset_index(drop=True)

    for index, record in tqdm(
        result.iterrows(),
        total=len(result),
        desc=f"{split_name} 배경 정리",
    ):
        with Image.open(record["crop_path"]) as source:
            rgb = np.asarray(source.convert("RGB"))

        mask = make_trial_mask(rgb)
        empty = not mask.any()

        # 마스크가 비면 원본을 유지하고 기록
        cleaned = rgb.copy()
        if not empty:
            cleaned[~mask] = 255

        case_dir = (
            mask_output_root
            / split_name
            / Path(record["source_image"]).stem
        )
        case_dir.mkdir(parents=True, exist_ok=True)

        object_id = int(record["object_index"])
        image_path = case_dir / f"object_{object_id:03d}.png"
        mask_path = case_dir / f"object_{object_id:03d}_mask.png"

        Image.fromarray(cleaned).save(image_path)
        Image.fromarray(mask.astype(np.uint8) * 255).save(mask_path)

        result.loc[index, "original_crop_path"] = record["crop_path"]
        result.loc[index, "crop_path"] = str(image_path)
        result.loc[index, "trial_mask_path"] = str(mask_path)
        result.loc[index, "mask_empty"] = empty
        result.loc[index, "mask_area_fraction"] = float(mask.mean())

    result.to_csv(
        mask_manifest_dir / f"{split_name}_crops.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(f"\n{split_name}: {len(result):,}개 저장")
    print("빈 마스크로 원본을 유지한 조각:",
          int(result["mask_empty"].sum()), "개")
    return result


masked_train_crops = save_masked_crops(
    expanded_train_crops, "train"
)
masked_validation_crops = save_masked_crops(
    expanded_validation_crops, "validation"
)

print("\n저장 위치:", mask_output_root)
print("배경 정리 완료 — 특징 추출·분류기 학습 전")

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm

masked_feature_dir = (
    BASE / "features" / "number_background_mask_v1"
)
masked_feature_dir.mkdir(parents=True, exist_ok=True)


def extract_masked_features(table, split_name):
    table = table.reset_index(drop=True)
    batches = []

    tvg_number_encoder.eval()

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(table), 4),
            desc=f"{split_name} 배경 정리 특징",
        ):
            images = []

            for path in table.iloc[start:start + 4]["crop_path"]:
                with Image.open(path) as source:
                    images.append(
                        tvg_transform(source.convert("RGB"))
                    )

            inputs = torch.stack(images).to(device)
            features = tvg_number_encoder(inputs).cpu()

            if not torch.isfinite(features).all().item():
                raise ValueError("특징에 비정상 값이 있어.")

            batches.append(features)

    features = torch.cat(batches, dim=0)
    assert features.shape == (len(table), 1024)

    payload = {
        "features": features,
        "labels": torch.tensor(
            table["target"].to_numpy(), dtype=torch.long
        ),
        "crop_paths": table["crop_path"].tolist(),
        "original_crop_paths": table["original_crop_path"].tolist(),
        "source_images": table["source_image"].tolist(),
        "object_indices": table["object_index"].tolist(),
        "data_cohorts": table["data_cohort"].tolist(),
        "chromosome_names": number_names,
        "metadata": {
            **expanded_train_features["metadata"],
            "preprocessing": str(tvg_transform),
            "experiment": "background_mask_v1",
            "background_method": (
                "gaussian_sigma_0.6_otsu_largest_component_8connect"
            ),
            "empty_mask_policy": "keep_original",
            "mask_manifest_directory": str(mask_manifest_dir),
        },
    }

    destination = masked_feature_dir / f"{split_name}_features.pt"
    temporary = destination.with_suffix(".pt.part")

    torch.save(payload, temporary)
    temporary.replace(destination)

    print(f"\n{split_name} 특징 크기:", tuple(features.shape))
    print("저장 위치:", destination)
    return payload


masked_train_features = extract_masked_features(
    masked_train_crops, "train"
)
masked_validation_features = extract_masked_features(
    masked_validation_crops, "validation"
)

print("\n배경 정리 특징 저장 완료 — 아직 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

masked_X_train = masked_train_features["features"].numpy()
masked_y_train = masked_train_features["labels"].numpy()
masked_X_val = masked_validation_features["features"].numpy()
masked_y_val = masked_validation_features["labels"].numpy()

# 같은 원본 조각·정답 순서인지 확인
for masked, original in [
    (masked_train_features, expanded_train_features),
    (masked_validation_features, expanded_validation_features),
]:
    assert masked["original_crop_paths"] == original["crop_paths"]
    assert masked["source_images"] == original["source_images"]
    assert masked["object_indices"] == original["object_indices"]
    assert np.array_equal(
        masked["labels"].numpy(), original["labels"].numpy()
    )
    assert (
        masked["metadata"]["preprocessing"]
        == original["metadata"]["preprocessing"]
    )

masked_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.01,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("배경 정리 분류기 학습 시작")
masked_classifier.fit(masked_X_train, masked_y_train)

cohort_array = np.array(masked_validation_features["data_cohorts"])

groups = {
    "전체 57장": np.ones(len(masked_y_val), dtype=bool),
    "기존 20장": cohort_array == "original_abnormal_cases",
    "추가 37장": cohort_array == "additional_normal_cases",
}

rows = []
masked_val_predictions = None

for model_name, classifier, features in [
    ("배경 정리 전", expanded_classifier, expanded_X_val),
    ("배경 정리 후", masked_classifier, masked_X_val),
]:
    predictions = classifier.predict(features)

    if model_name == "배경 정리 후":
        masked_val_predictions = predictions

    for group_name, mask in groups.items():
        actual = masked_y_val[mask]
        predicted = predictions[mask]

        actual_21 = actual == number_to_id["chr21"]
        predicted_21 = predicted == number_to_id["chr21"]

        photo_counts = pd.DataFrame({
            "source_image": np.array(
                masked_validation_features["source_images"]
            )[mask],
            "actual": actual_21.astype(int),
            "predicted": predicted_21.astype(int),
            "fp": (~actual_21 & predicted_21).astype(int),
            "fn": (actual_21 & ~predicted_21).astype(int),
        }).groupby("source_image").sum()

        rows.append({
            "모델": model_name,
            "검증 자료": group_name,
            "번호 정확도": accuracy_score(actual, predicted),
            "Macro F1": f1_score(
                actual, predicted,
                labels=list(range(24)),
                average="macro",
                zero_division=0,
            ),
            "21번 개수 일치 사진": int(
                photo_counts["actual"].eq(
                    photo_counts["predicted"]
                ).sum()
            ),
            "21번 식별 모두 정확한 사진": int(
                (
                    photo_counts["fp"].eq(0)
                    & photo_counts["fn"].eq(0)
                ).sum()
            ),
            "21번 오분류": int(photo_counts["fp"].sum()),
            "21번 놓침": int(photo_counts["fn"].sum()),
        })

mask_comparison = pd.DataFrame(rows)
display(mask_comparison.round(4))

# 기존 모델과 별도로 저장
masked_model_dir = BASE / "models" / "number_background_mask_linear_v1"
masked_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": masked_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": masked_train_features["metadata"],
    "C": 0.01,
}, masked_model_dir / "classifier.joblib")

mask_comparison.to_csv(
    masked_model_dir / "validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

masked_validation_results = masked_validation_crops.copy()
masked_validation_results["predicted_chromosome"] = [
    number_names[int(i)] for i in masked_val_predictions
]
masked_validation_results.to_csv(
    masked_model_dir / "validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n학습·비교·저장 완료:", masked_model_dir)

In [ ]:
review = masked_validation_results.copy()

# 실제 21번인데 다른 번호로 예측
missed_21 = review.loc[
    review["chromosome"].eq("chr21")
    & ~review["predicted_chromosome"].eq("chr21")
]

# 다른 번호인데 21번으로 예측
false_21 = review.loc[
    ~review["chromosome"].eq("chr21")
    & review["predicted_chromosome"].eq("chr21")
]

examples = []

for error_name, table in [
    ("Missed chr21", missed_21),
    ("False chr21", false_21),
]:
    # 여러 원본 사진에서 예시 선택
    candidates = (
        table.sort_values(["source_image", "object_index"])
        .drop_duplicates("source_image")
    )
    selected = candidates.sample(
        n=min(3, len(candidates)),
        random_state=42,
    )

    for _, record in selected.iterrows():
        examples.append((error_name, record))

fig, axes = plt.subplots(
    len(examples), 2,
    figsize=(8, 3 * len(examples)),
    squeeze=False,
)

for row_index, (error_name, record) in enumerate(examples):
    for column, path in enumerate([
        record["original_crop_path"],
        record["crop_path"],
    ]):
        with Image.open(path) as source:
            axes[row_index, column].imshow(source.convert("RGB"))
        axes[row_index, column].axis("off")

    axes[row_index, 0].set_title(
        f"Original: {record['chromosome']}\n"
        f"{record['source_image']}",
        fontsize=9,
    )
    axes[row_index, 1].set_title(
        f"{error_name}\n"
        f"Masked prediction: {record['predicted_chromosome']}",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

local_view_root = BASE / "derived" / "number_local_view_v1"
local_manifest_dir = BASE / "splits" / "number_local_view_v1"
local_manifest_dir.mkdir(parents=True, exist_ok=True)


def create_local_views(table, split_name):
    result = table.copy().reset_index(drop=True)

    for index, record in tqdm(
        result.iterrows(),
        total=len(result),
        desc=f"{split_name} 확대용 사진",
    ):
        with Image.open(record["crop_path"]) as source:
            image = source.convert("RGB")

        with Image.open(record["trial_mask_path"]) as source:
            mask = np.asarray(source.convert("L")) > 0

        assert mask.shape == (image.height, image.width)

        ys, xs = np.where(mask)

        if len(xs) == 0:
            # 빈 마스크라면 현재 화면 전체 유지
            local_crop = image.copy()
        else:
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1

            margin = max(
                2, int(np.ceil(max(x2 - x1, y2 - y1) * 0.08))
            )

            box = (
                max(0, x1 - margin),
                max(0, y1 - margin),
                min(image.width, x2 + margin),
                min(image.height, y2 + margin),
            )
            local_crop = image.crop(box)

        # 가로세로 비율을 유지하도록 정사각형 바탕에 배치
        side = max(local_crop.size)
        canvas = Image.new("RGB", (side, side), "white")
        canvas.paste(
            local_crop,
            (
                (side - local_crop.width) // 2,
                (side - local_crop.height) // 2,
            ),
        )

        case_dir = (
            local_view_root
            / split_name
            / Path(record["source_image"]).stem
        )
        case_dir.mkdir(parents=True, exist_ok=True)

        destination = (
            case_dir / f"object_{int(record['object_index']):03d}.png"
        )
        canvas.save(destination)

        # 기존 화면 경로도 함께 보존
        result.loc[index, "global_crop_path"] = record["crop_path"]
        result.loc[index, "local_crop_path"] = str(destination)

    result.to_csv(
        local_manifest_dir / f"{split_name}_views.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(f"{split_name}: {len(result):,}개 저장")
    return result


dual_train_views = create_local_views(masked_train_crops, "train")
dual_validation_views = create_local_views(
    masked_validation_crops, "validation"
)

print("\n확대용 사진 준비 완료 — 특징 추출·학습 전")
print("저장 위치:", local_view_root)

In [ ]:
import torch
from PIL import Image
from tqdm.auto import tqdm

dual_feature_dir = BASE / "features" / "number_dual_view_v1"
dual_feature_dir.mkdir(parents=True, exist_ok=True)


def extract_dual_features(table, global_payload, split_name):
    table = table.reset_index(drop=True)

    # 기존 화면 특징과 확대용 사진의 순서 확인
    assert table["global_crop_path"].tolist() == global_payload["crop_paths"]
    assert table["source_image"].tolist() == global_payload["source_images"]
    assert table["object_index"].tolist() == global_payload["object_indices"]
    assert torch.equal(
        torch.tensor(table["target"].to_numpy(), dtype=torch.long),
        global_payload["labels"],
    )
    assert (
        str(tvg_transform)
        == global_payload["metadata"]["preprocessing"]
    )

    local_batches = []
    tvg_number_encoder.eval()

    with torch.inference_mode():
        for start in tqdm(
            range(0, len(table), 4),
            desc=f"{split_name} 확대 특징 추출",
        ):
            images = []

            for path in table.iloc[start:start + 4]["local_crop_path"]:
                with Image.open(path) as source:
                    images.append(
                        tvg_transform(source.convert("RGB"))
                    )

            inputs = torch.stack(images).to(device)
            features = tvg_number_encoder(inputs).cpu()

            if not torch.isfinite(features).all().item():
                raise ValueError("확대 특징에 비정상 값이 있어.")

            local_batches.append(features)

    local_features = torch.cat(local_batches, dim=0)
    assert local_features.shape == (len(table), 1024)

    # 열 방향으로 연결: 앞 1024개는 기존 화면, 뒤는 확대 화면
    combined_features = torch.cat(
        [global_payload["features"], local_features],
        dim=1,
    )

    assert combined_features.shape == (len(table), 2048)
    assert torch.isfinite(combined_features).all().item()

    payload = {
        "features": combined_features,
        "labels": global_payload["labels"].clone(),
        "global_crop_paths": table["global_crop_path"].tolist(),
        "local_crop_paths": table["local_crop_path"].tolist(),
        "source_images": table["source_image"].tolist(),
        "object_indices": table["object_index"].tolist(),
        "data_cohorts": table["data_cohort"].tolist(),
        "chromosome_names": number_names,
        "metadata": {
            **global_payload["metadata"],
            "experiment": "global_and_local_view_v1",
            "feature_order": ["global_1024", "local_1024"],
            "feature_dimension": 2048,
            "local_view_method": (
                "trial_mask_bbox_margin_8percent_min2_square_pad"
            ),
            "local_manifest_directory": str(local_manifest_dir),
        },
    }

    destination = dual_feature_dir / f"{split_name}_features.pt"
    temporary = destination.with_suffix(".pt.part")

    torch.save(payload, temporary)
    temporary.replace(destination)

    print(f"\n{split_name} 통합 특징 크기:",
          tuple(combined_features.shape))
    print("저장 위치:", destination)

    return payload


dual_train_features = extract_dual_features(
    dual_train_views,
    masked_train_features,
    "train",
)

dual_validation_features = extract_dual_features(
    dual_validation_views,
    masked_validation_features,
    "validation",
)

print("\n기존 화면 + 확대 화면 특징 저장 완료")
print("아직 새 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

dual_X_train = dual_train_features["features"].numpy()
dual_y_train = dual_train_features["labels"].numpy()
dual_X_val = dual_validation_features["features"].numpy()
dual_y_val = dual_validation_features["labels"].numpy()

# 기존 화면 특징과 정답·순서가 같은지 확인
for combined, original in [
    (dual_train_features, masked_train_features),
    (dual_validation_features, masked_validation_features),
]:
    assert combined["source_images"] == original["source_images"]
    assert combined["object_indices"] == original["object_indices"]
    assert np.array_equal(
        combined["labels"].numpy(),
        original["labels"].numpy(),
    )
    assert np.array_equal(
        combined["features"][:, :1024].numpy(),
        original["features"].numpy(),
    )

dual_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.01,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("기존 화면 + 확대 화면 분류기 학습 시작")
dual_classifier.fit(dual_X_train, dual_y_train)

cohort_array = np.array(dual_validation_features["data_cohorts"])

groups = {
    "전체 57장": np.ones(len(dual_y_val), dtype=bool),
    "기존 20장": cohort_array == "original_abnormal_cases",
    "추가 37장": cohort_array == "additional_normal_cases",
}

rows = []
dual_val_predictions = None

for model_name, classifier, features in [
    ("기존 화면만", masked_classifier, masked_X_val),
    ("기존 + 확대", dual_classifier, dual_X_val),
]:
    predictions = classifier.predict(features)

    if model_name == "기존 + 확대":
        dual_val_predictions = predictions

    for group_name, mask in groups.items():
        actual = dual_y_val[mask]
        predicted = predictions[mask]

        actual_21 = actual == number_to_id["chr21"]
        predicted_21 = predicted == number_to_id["chr21"]

        photo_counts = pd.DataFrame({
            "source_image": np.array(
                dual_validation_features["source_images"]
            )[mask],
            "actual": actual_21.astype(int),
            "predicted": predicted_21.astype(int),
            "fp": (~actual_21 & predicted_21).astype(int),
            "fn": (actual_21 & ~predicted_21).astype(int),
        }).groupby("source_image").sum()

        rows.append({
            "모델": model_name,
            "검증 자료": group_name,
            "번호 정확도": accuracy_score(actual, predicted),
            "Macro F1": f1_score(
                actual, predicted,
                labels=list(range(24)),
                average="macro",
                zero_division=0,
            ),
            "21번 개수 일치 사진": int(
                photo_counts["actual"].eq(
                    photo_counts["predicted"]
                ).sum()
            ),
            "21번 식별 모두 정확한 사진": int(
                (
                    photo_counts["fp"].eq(0)
                    & photo_counts["fn"].eq(0)
                ).sum()
            ),
            "21번 오분류": int(photo_counts["fp"].sum()),
            "21번 놓침": int(photo_counts["fn"].sum()),
        })

dual_comparison = pd.DataFrame(rows)
display(dual_comparison.round(4))

dual_model_dir = BASE / "models" / "number_dual_view_linear_v1"
dual_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": dual_classifier,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": dual_train_features["metadata"],
    "C": 0.01,
}, dual_model_dir / "classifier.joblib")

dual_comparison.to_csv(
    dual_model_dir / "validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

dual_validation_results = dual_validation_views.copy()
dual_validation_results["predicted_chromosome"] = [
    number_names[int(i)] for i in dual_val_predictions
]
dual_validation_results.to_csv(
    dual_model_dir / "validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n학습·비교·저장 완료:", dual_model_dir)

In [ ]:
# 학습·검증 사진과 테스트 파일명이 겹치지 않는지 확인
test_names = set(expanded_test_cases["filename"])
development_names = (
    set(expanded_train_cases["filename"])
    | set(expanded_validation_cases["filename"])
)
assert not (test_names & development_names)

# 1. 원본 사진별 공통 바탕으로 조각 추출
final_test_crops = extract_number_crops(
    expanded_test_cases, "test_final_v1"
)

# 원본 사진의 출처 연결
final_test_crops = final_test_crops.merge(
    expanded_test_cases[["filename", "data_cohort"]],
    left_on="source_image",
    right_on="filename",
    how="left",
    validate="many_to_one",
).drop(columns=["filename"])

assert final_test_crops["data_cohort"].notna().all()

# 2. 학습·검증과 동일한 배경 정리
final_masked_test_crops = save_masked_crops(
    final_test_crops, "test_final_v1"
)

# 3. 동일한 방식으로 확대용 화면 생성
final_dual_test_views = create_local_views(
    final_masked_test_crops, "test_final_v1"
)

print("\n===== 테스트 입력 준비 완료 =====")
print(
    "원본 사진:",
    final_dual_test_views["source_image"].nunique(), "장"
)
print("염색체 조각:", len(final_dual_test_views), "개")

display(
    final_dual_test_views.groupby("data_cohort").agg(
        원본_사진_수=("source_image", "nunique"),
        조각_수=("object_index", "size"),
    )
)

print("아직 테스트 예측 전 — 재학습하지 않았어.")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report
)

# 1. 저장된 최종 평가 후보 모델 불러오기
final_bundle = joblib.load(
    dual_model_dir / "classifier.joblib"
)
final_classifier = final_bundle["classifier"]

assert final_bundle["chromosome_names"] == number_names

# 2. 기존 화면 + 확대 화면 특징 추출
final_global_test_features = extract_masked_features(
    final_masked_test_crops, "test_final_v1"
)

final_dual_test_features = extract_dual_features(
    final_dual_test_views,
    final_global_test_features,
    "test_final_v1",
)

# 학습 때와 같은 특징 생성 설정인지 확인
assert (
    final_dual_test_features["metadata"]
    == final_bundle["feature_metadata"]
)

# 3. 모델을 변경하지 않고 예측
final_X_test = final_dual_test_features["features"].numpy()
final_y_test = final_dual_test_features["labels"].numpy()

final_test_predictions = final_classifier.predict(final_X_test)

final_test_results = final_dual_test_views.copy().reset_index(drop=True)
final_test_results["predicted_chromosome"] = [
    number_names[int(i)] for i in final_test_predictions
]
final_test_results["correct"] = (
    final_y_test == final_test_predictions
)

# 4. 사진별 21번 개수·식별 평가
count_records = []

for filename, group in final_test_results.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    count_records.append({
        "source_image": filename,
        "data_cohort": group["data_cohort"].iloc[0],
        "actual_chr21_count": int(actual.sum()),
        "predicted_chr21_count": int(predicted.sum()),
        "false_positive": fp,
        "false_negative": fn,
        "count_correct": int(actual.sum()) == int(predicted.sum()),
        "identification_correct": fp == 0 and fn == 0,
    })

final_count_review = pd.DataFrame(count_records)

# 5. 전체·출처별 결과 요약
cohort_array = final_test_results["data_cohort"].to_numpy()
groups = [
    ("전체 58장", None),
    ("기존 평가 자료 21장", "original_abnormal_cases"),
    ("처음 평가하는 추가 정상 37장", "additional_normal_cases"),
]

rows = []

for title, cohort in groups:
    mask = (
        np.ones(len(final_y_test), dtype=bool)
        if cohort is None else cohort_array == cohort
    )

    photo_review = (
        final_count_review
        if cohort is None else
        final_count_review.loc[
            final_count_review["data_cohort"].eq(cohort)
        ]
    )

    rows.append({
        "평가 자료": title,
        "조각 수": int(mask.sum()),
        "번호 정확도": accuracy_score(
            final_y_test[mask], final_test_predictions[mask]
        ),
        "Macro F1": f1_score(
            final_y_test[mask], final_test_predictions[mask],
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
        "21번 개수 일치 사진": int(
            photo_review["count_correct"].sum()
        ),
        "21번 식별 모두 정확한 사진": int(
            photo_review["identification_correct"].sum()
        ),
        "21번 오분류": int(photo_review["false_positive"].sum()),
        "21번 놓침": int(photo_review["false_negative"].sum()),
    })

final_test_metrics = pd.DataFrame(rows)
print("\n===== 고정 모델 테스트 결과 =====")
display(final_test_metrics.round(4))

# 6. 결과 저장
final_evaluation_dir = dual_model_dir / "test_evaluation_v1"
final_evaluation_dir.mkdir(parents=True, exist_ok=True)

final_test_metrics.to_csv(
    final_evaluation_dir / "metrics.csv",
    index=False, encoding="utf-8-sig",
)
final_test_results.to_csv(
    final_evaluation_dir / "predictions.csv",
    index=False, encoding="utf-8-sig",
)
final_count_review.to_csv(
    final_evaluation_dir / "chr21_count_review.csv",
    index=False, encoding="utf-8-sig",
)

report = classification_report(
    final_y_test, final_test_predictions,
    labels=list(range(24)),
    target_names=number_names,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.to_csv(
    final_evaluation_dir / "classification_report.csv",
    encoding="utf-8-sig",
)

print("\n테스트 평가·저장 완료:", final_evaluation_dir)
print("모델을 재학습하거나 변경하지 않았어.")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# 이미 학습한 C=0.01 모델은 재사용
dual_reg_classifiers = {0.01: dual_classifier}
rows = []

for c_value in [0.001, 0.01, 0.1, 1.0]:
    print(f"C={c_value} 학습·검증 중")

    if c_value not in dual_reg_classifiers:
        classifier = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=c_value,
                max_iter=3000,
                solver="lbfgs",
                random_state=42,
            ),
        )
        classifier.fit(dual_X_train, dual_y_train)
        dual_reg_classifiers[c_value] = classifier

    classifier = dual_reg_classifiers[c_value]

    train_pred = classifier.predict(dual_X_train)
    val_pred = classifier.predict(dual_X_val)

    actual_21 = dual_y_val == number_to_id["chr21"]
    predicted_21 = val_pred == number_to_id["chr21"]

    photo_counts = pd.DataFrame({
        "source_image": dual_validation_features["source_images"],
        "actual": actual_21.astype(int),
        "predicted": predicted_21.astype(int),
        "fp": (~actual_21 & predicted_21).astype(int),
        "fn": (actual_21 & ~predicted_21).astype(int),
    }).groupby("source_image").sum()

    rows.append({
        "C": c_value,
        "학습 정확도": accuracy_score(dual_y_train, train_pred),
        "검증 정확도": accuracy_score(dual_y_val, val_pred),
        "검증 Macro F1": f1_score(
            dual_y_val, val_pred,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
        "21번 개수 일치 사진": int(
            photo_counts["actual"].eq(
                photo_counts["predicted"]
            ).sum()
        ),
        "21번 식별 모두 정확한 사진": int(
            (
                photo_counts["fp"].eq(0)
                & photo_counts["fn"].eq(0)
            ).sum()
        ),
        "21번 오분류": int(photo_counts["fp"].sum()),
        "21번 놓침": int(photo_counts["fn"].sum()),
    })

dual_reg_comparison = pd.DataFrame(rows)
display(dual_reg_comparison.round(4))

dual_reg_comparison.to_csv(
    dual_model_dir / "regularization_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("비교 완료 — 기존 저장 모델은 변경하지 않았어.")
print("테스트 자료는 이번 비교에 사용하지 않았어.")

In [ ]:
import copy
import numpy as np
import torch
from torch import nn
from PIL import Image


class TVGDualFineTune(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(2048, 24)

    def forward(self, global_images, local_images):
        batch_size = len(global_images)

        # 같은 인코더로 두 화면 처리
        inputs = torch.cat([global_images, local_images], dim=0)
        features = self.encoder(inputs)

        global_features = features[:batch_size]
        local_features = features[batch_size:]

        combined = torch.cat(
            [global_features, local_features], dim=1
        )
        return self.head(combined)


# 이미 준비된 모델이 있다면 재사용하지 않도록 중단
if "dual_finetune_model" in globals():
    raise RuntimeError(
        "dual_finetune_model이 이미 있어. "
        "이 셀을 다시 실행하지 말고 현재 상태를 알려줘."
    )

dual_finetune_model = TVGDualFineTune(
    copy.deepcopy(tvg_number_encoder)
).to(device)

# C=0.1 분류기를 새 분류층으로 변환
start_classifier = dual_reg_classifiers[0.1]
scaler = start_classifier.named_steps["standardscaler"]
linear = start_classifier.named_steps["logisticregression"]

assert np.array_equal(linear.classes_, np.arange(24))

weight = linear.coef_ / scaler.scale_[None, :]
bias = linear.intercept_ - weight @ scaler.mean_

with torch.no_grad():
    dual_finetune_model.head.weight.copy_(
        torch.tensor(weight, dtype=torch.float32, device=device)
    )
    dual_finetune_model.head.bias.copy_(
        torch.tensor(bias, dtype=torch.float32, device=device)
    )

# 전체 동결 후 마지막 2개 블록·정규화·분류층만 학습 허용
dual_finetune_model.requires_grad_(False)

for module in [
    dual_finetune_model.encoder.blocks[-2],
    dual_finetune_model.encoder.blocks[-1],
    dual_finetune_model.encoder.fc_norm,
    dual_finetune_model.head,
]:
    module.requires_grad_(True)

# 출발 예측이 기존 분류기와 같은지 확인
global_images = []
local_images = []

for _, record in dual_validation_views.iloc[:2].iterrows():
    with Image.open(record["global_crop_path"]) as source:
        global_images.append(tvg_transform(source.convert("RGB")))

    with Image.open(record["local_crop_path"]) as source:
        local_images.append(tvg_transform(source.convert("RGB")))

dual_finetune_model.eval()

with torch.inference_mode():
    new_predictions = dual_finetune_model(
        torch.stack(global_images).to(device),
        torch.stack(local_images).to(device),
    ).argmax(1).cpu().numpy()

old_predictions = start_classifier.predict(dual_X_val[:2])
assert np.array_equal(new_predictions, old_predictions)

dual_finetune_dir = (
    BASE / "models" / "number_dual_view_finetune_v1"
)
dual_finetune_dir.mkdir(parents=True, exist_ok=True)

trainable = sum(
    p.numel() for p in dual_finetune_model.parameters()
    if p.requires_grad
)

print("출발 분류기: C=0.1")
print("출발 검증 정확도: 81.15%")
print("기존 분류기와 시험 입력의 예측 일치: True")
print(f"학습 가능한 파라미터: {trainable:,}개")
print("학습 범위: 마지막 2개 블록 + fc_norm + 분류층")
print("저장 예정 위치:", dual_finetune_dir)
print("두 화면 미세조정 준비 완료 — 아직 학습 전")

In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


class DualViewDataset(Dataset):
    def __init__(self, table, augment=False):
        self.table = table.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        record = self.table.iloc[index]

        with Image.open(record["global_crop_path"]) as source:
            global_image = source.convert("RGB")

        with Image.open(record["local_crop_path"]) as source:
            local_image = source.convert("RGB")

        if self.augment:
            # 두 화면에 동일한 변환 적용
            rotation = random.choice([
                None,
                Image.Transpose.ROTATE_90,
                Image.Transpose.ROTATE_180,
                Image.Transpose.ROTATE_270,
            ])

            if rotation is not None:
                global_image = global_image.transpose(rotation)
                local_image = local_image.transpose(rotation)

            if random.random() < 0.5:
                global_image = global_image.transpose(
                    Image.Transpose.FLIP_LEFT_RIGHT
                )
                local_image = local_image.transpose(
                    Image.Transpose.FLIP_LEFT_RIGHT
                )

        return (
            tvg_transform(global_image),
            tvg_transform(local_image),
            int(record["target"]),
        )


dual_ft_train_loader = DataLoader(
    DualViewDataset(dual_train_views, augment=True),
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

dual_ft_val_loader = DataLoader(
    DualViewDataset(dual_validation_views, augment=False),
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

criterion = nn.CrossEntropyLoss()

# 인코더는 더 작은 학습률로 조심스럽게 변경
dual_optimizer = torch.optim.AdamW([
    {
        "params": [
            p for p in dual_finetune_model.encoder.parameters()
            if p.requires_grad
        ],
        "lr": 2e-6,
    },
    {
        "params": dual_finetune_model.head.parameters(),
        "lr": 1e-5,
    },
], weight_decay=1e-4)

trainable_parameters = [
    p for p in dual_finetune_model.parameters()
    if p.requires_grad
]


def evaluate_dual_finetune():
    dual_finetune_model.eval()
    actual, predicted = [], []
    loss_sum = 0.0

    with torch.inference_mode():
        for global_images, local_images, labels in tqdm(
            dual_ft_val_loader, desc="검증"
        ):
            labels = labels.to(device)

            logits = dual_finetune_model(
                global_images.to(device),
                local_images.to(device),
            )
            loss = criterion(logits, labels)

            loss_sum += loss.item() * len(labels)
            actual.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(1).cpu().tolist())

    return {
        "loss": float(loss_sum / len(actual)),
        "accuracy": float(accuracy_score(actual, predicted)),
        "macro_f1": float(f1_score(
            actual, predicted,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        )),
    }


def save_dual_best(epoch, metrics):
    destination = dual_finetune_dir / "best.pth"
    temporary = destination.with_suffix(".pth.part")

    torch.save({
        "model": {
            key: value.detach().cpu()
            for key, value in dual_finetune_model.state_dict().items()
        },
        "epoch": epoch,
        "validation_metrics": metrics,
        "architecture": "shared_vit_large_patch16_dual_view",
        "global_pool": "avg",
        "chromosome_names": number_names,
        "feature_metadata": dual_train_features["metadata"],
        "training_augmentation": (
            "synchronized_0_90_180_270_rotation_and_horizontal_flip"
        ),
        "selection_metric": "validation_macro_f1",
    }, temporary)

    temporary.replace(destination)


# 出発モデルを評価・保存
baseline = evaluate_dual_finetune()
best_f1 = baseline["macro_f1"]
best_epoch = 0
no_improvement = 0

save_dual_best(0, baseline)

history = [{
    "epoch": 0,
    "train_loss": None,
    "train_accuracy": None,
    "validation_loss": baseline["loss"],
    "validation_accuracy": baseline["accuracy"],
    "validation_macro_f1": baseline["macro_f1"],
}]

pd.DataFrame(history).to_csv(
    dual_finetune_dir / "history.csv",
    index=False, encoding="utf-8-sig",
)

print(
    f"\n출발 성능 | 정확도 {baseline['accuracy']:.2%}"
    f" | Macro F1 {baseline['macro_f1']:.4f}"
)

# 배치 2개씩 8회 누적 → 16개 기준으로 가중치 갱신
accumulation_steps = 8

for epoch in range(1, 7):
    # 고정된 앞부분은 평가 모드 유지
    dual_finetune_model.eval()

    for block in dual_finetune_model.encoder.blocks[-2:]:
        block.train()

    dual_finetune_model.encoder.fc_norm.train()
    dual_finetune_model.head.train()

    dual_optimizer.zero_grad(set_to_none=True)

    loss_sum = 0.0
    correct = 0
    total = 0
    loader_length = len(dual_ft_train_loader)

    for batch_index, (global_images, local_images, labels) in enumerate(
        tqdm(dual_ft_train_loader, desc=f"학습 {epoch}/6")
    ):
        labels = labels.to(device)

        logits = dual_finetune_model(
            global_images.to(device),
            local_images.to(device),
        )
        loss = criterion(logits, labels)

        if not torch.isfinite(loss).item():
            raise RuntimeError("Loss가 비정상이어서 중단했어.")

        # 마지막 누적 묶음의 크기도 반영
        group_start = (
            batch_index // accumulation_steps
        ) * accumulation_steps
        group_size = min(
            accumulation_steps, loader_length - group_start
        )

        (loss / group_size).backward()

        update_now = (
            (batch_index + 1) % accumulation_steps == 0
            or batch_index + 1 == loader_length
        )
        if update_now:
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters, max_norm=1.0
            )
            dual_optimizer.step()
            dual_optimizer.zero_grad(set_to_none=True)

        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    metrics = evaluate_dual_finetune()
    improved = metrics["macro_f1"] > best_f1 + 1e-6

    if improved:
        best_f1 = metrics["macro_f1"]
        best_epoch = epoch
        no_improvement = 0
        save_dual_best(epoch, metrics)
    else:
        no_improvement += 1

    history.append({
        "epoch": epoch,
        "train_loss": loss_sum / total,
        "train_accuracy": correct / total,
        "validation_loss": metrics["loss"],
        "validation_accuracy": metrics["accuracy"],
        "validation_macro_f1": metrics["macro_f1"],
    })

    pd.DataFrame(history).to_csv(
        dual_finetune_dir / "history.csv",
        index=False, encoding="utf-8-sig",
    )

    print(
        f"\nEpoch {epoch}/6"
        f" | 증강 학습 정확도 {correct / total:.2%}"
        f" | 검증 정확도 {metrics['accuracy']:.2%}"
        f" | 검증 Macro F1 {metrics['macro_f1']:.4f}"
        + (" ★ 최적 모델 저장" if improved else "")
    )

    if no_improvement >= 3:
        print("검증 Macro F1이 3에포크 연속 개선되지 않아 중단했어.")
        break

dual_finetune_model.eval()

print("\n학습 완료")
print("최적 에포크:", best_epoch, "(0이면 출발 모델)")
print(f"최적 검증 Macro F1: {best_f1:.4f}")
print("저장 위치:", dual_finetune_dir)
print("현재 메모리에는 마지막 에포크 모델이 있어.")
print("다음 평가는 best.pth를 복원해서 진행할 거야.")

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

# 1. 최적 모델 복원
best_checkpoint = torch.load(
    dual_finetune_dir / "best.pth",
    map_location="cpu",
    weights_only=True,
    mmap=True,
)

dual_finetune_model.load_state_dict(
    best_checkpoint["model"], strict=True
)
dual_finetune_model.eval()

print("복원한 에포크:", best_checkpoint["epoch"])

# 2. 검증 예측
actual_labels = []
predicted_labels = []

with torch.inference_mode():
    for global_images, local_images, labels in tqdm(
        dual_ft_val_loader, desc="최적 모델 검증"
    ):
        logits = dual_finetune_model(
            global_images.to(device),
            local_images.to(device),
        )

        actual_labels.extend(labels.tolist())
        predicted_labels.extend(logits.argmax(1).cpu().tolist())

actual_labels = np.asarray(actual_labels)
predicted_labels = np.asarray(predicted_labels)

ft_review = dual_validation_views.copy().reset_index(drop=True)
assert np.array_equal(
    actual_labels, ft_review["target"].to_numpy()
)

ft_review["predicted_chromosome"] = [
    number_names[int(i)] for i in predicted_labels
]
ft_review["correct"] = actual_labels == predicted_labels

# 3. 写真ごとの21番の個数・識別を評価
photo_rows = []

for filename, group in ft_review.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    photo_rows.append({
        "source_image": filename,
        "data_cohort": group["data_cohort"].iloc[0],
        "actual_chr21_count": int(actual.sum()),
        "predicted_chr21_count": int(predicted.sum()),
        "false_positive": fp,
        "false_negative": fn,
        "count_correct": int(actual.sum()) == int(predicted.sum()),
        "identification_correct": fp == 0 and fn == 0,
    })

ft_photo_review = pd.DataFrame(photo_rows)

# 4. 全体・データ出典別に集計
rows = []
cohort_array = ft_review["data_cohort"].to_numpy()

for title, cohort in [
    ("전체 57장", None),
    ("기존 20장", "original_abnormal_cases"),
    ("추가 37장", "additional_normal_cases"),
]:
    mask = (
        np.ones(len(ft_review), dtype=bool)
        if cohort is None else cohort_array == cohort
    )

    photos = (
        ft_photo_review
        if cohort is None else
        ft_photo_review.loc[
            ft_photo_review["data_cohort"].eq(cohort)
        ]
    )

    rows.append({
        "검증 자료": title,
        "번호 정확도": accuracy_score(
            actual_labels[mask], predicted_labels[mask]
        ),
        "Macro F1": f1_score(
            actual_labels[mask], predicted_labels[mask],
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        ),
        "21번 개수 일치 사진": int(photos["count_correct"].sum()),
        "21번 식별 모두 정확한 사진": int(
            photos["identification_correct"].sum()
        ),
        "21번 오분류": int(photos["false_positive"].sum()),
        "21번 놓침": int(photos["false_negative"].sum()),
    })

ft_metrics = pd.DataFrame(rows)
display(ft_metrics.round(4))

ft_review.to_csv(
    dual_finetune_dir / "validation_predictions.csv",
    index=False, encoding="utf-8-sig",
)
ft_photo_review.to_csv(
    dual_finetune_dir / "validation_chr21_review.csv",
    index=False, encoding="utf-8-sig",
)
ft_metrics.to_csv(
    dual_finetune_dir / "validation_metrics.csv",
    index=False, encoding="utf-8-sig",
)

print("최적 모델 검증·저장 완료 — 테스트 평가 없음")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm.auto import tqdm

selection_path = BASE / "normal_additional_1000_selected_v1.csv"
acquisition_path = BASE / "normal_additional_1000_acquisition_v1.csv"

# 이미 선택 목록을 만들었다면 같은 목록 재사용
if selection_path.is_file():
    additional_names = pd.read_csv(
        selection_path
    )["filename"].tolist()

else:
    # 이전 다운로드 시도 200장과 기존 사례를 모두 제외
    excluded_names = (
        set(normal_acquired["filename"].astype(str))
        | set(catalog["filename"].astype(str))
        | set(expanded_train_cases["filename"].astype(str))
        | set(expanded_validation_cases["filename"].astype(str))
        | set(expanded_test_cases["filename"].astype(str))
    )

    candidates = (
        normal_names.loc[~normal_names.isin(excluded_names)]
        .sort_values()
        .reset_index(drop=True)
    )

    assert len(candidates) >= 1000

    additional_names = candidates.sample(
        n=1000, random_state=42
    ).tolist()

    pd.DataFrame({"filename": additional_names}).to_csv(
        selection_path,
        index=False,
        encoding="utf-8-sig",
    )

# 검증·테스트 사진이 다운로드 대상에 포함되지 않았는지 확인
held_out_names = (
    set(expanded_validation_cases["filename"])
    | set(expanded_test_cases["filename"])
)
assert not (set(additional_names) & held_out_names)

# 이전에 성공한 기록은 재사용
if acquisition_path.is_file():
    previous = pd.read_csv(acquisition_path, keep_default_na=False)
    successful_previous = previous.loc[
        previous["acquisition_status"].eq("paired")
    ]

    # 실제 두 파일도 남아 있는 사례만 완료 처리
    results = [
        record
        for record in successful_previous.to_dict("records")
        if (BASE / record["image_path"]).is_file()
        and (BASE / record["annotation_path"]).is_file()
    ]
else:
    results = []

completed_names = {record["filename"] for record in results}
pending_names = [
    name for name in additional_names
    if name not in completed_names
]

print("이번 선택:", len(additional_names), "장")
print("이전 성공 기록 재사용:", len(results), "쌍")
print("다운로드·확인할 자료:", len(pending_names), "쌍")

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [
        pool.submit(acquire_normal_case, filename)
        for filename in pending_names
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="추가 1000장 사진·주석 확보",
    ):
        results.append(future.result())

        pd.DataFrame(results).to_csv(
            acquisition_path,
            index=False,
            encoding="utf-8-sig",
        )

additional_1000_acquired = pd.DataFrame(results)
paired_mask = additional_1000_acquired[
    "acquisition_status"
].eq("paired")

print("\n확보:", int(paired_mask.sum()), "쌍")
print("실패:", int((~paired_mask).sum()), "건")

print("\n정답 상자 수 분포:")
display(
    pd.to_numeric(
        additional_1000_acquired.loc[
            paired_mask, "xml_object_count"
        ]
    )
    .value_counts()
    .sort_index()
    .rename_axis("정답 상자 수")
    .to_frame("사진 수")
)

if (~paired_mask).any():
    display(
        additional_1000_acquired.loc[
            ~paired_mask, ["filename", "error"]
        ]
    )

print("저장 위치:", acquisition_path)
print("자료 확보 완료 — 아직 주석 검사·학습 전")

In [ ]:
from collections import Counter
import xml.etree.ElementTree as ET
import pandas as pd
import re
from tqdm.auto import tqdm

acquired_cases = additional_1000_acquired.loc[
    additional_1000_acquired["acquisition_status"].eq("paired")
].copy()

assert acquired_cases["filename"].is_unique

audit_records = []

for _, case in tqdm(
    acquired_cases.iterrows(),
    total=len(acquired_cases),
    desc="추가 확보 자료 주석 검사",
):
    root = ET.parse(BASE / case["annotation_path"]).getroot()

    width = int(root.findtext("size/width"))
    height = int(root.findtext("size/height"))
    objects = root.findall("object")

    counts = Counter()
    unknown_labels = []
    invalid_boxes = 0
    outside_boxes = 0

    for obj in objects:
        label = (obj.findtext("name") or "").strip()

        if label in ("X", "Y"):
            chromosome = f"chr{label}"
        else:
            match = re.fullmatch(r"[A-G](\d+)", label)
            chromosome = (
                f"chr{int(match.group(1))}" if match else None
            )

        if chromosome not in number_to_id:
            unknown_labels.append(label)
        else:
            counts[chromosome] += 1

        x1, y1, x2, y2 = [
            int(obj.findtext(f"bndbox/{key}"))
            for key in ["xmin", "ymin", "xmax", "ymax"]
        ]

        invalid_boxes += int(x2 <= x1 or y2 <= y1)
        outside_boxes += int(
            x1 < 0 or y1 < 0 or x2 > width or y2 > height
        )

    reasons = []

    if len(objects) != 46:
        reasons.append("상자 총수 46개 아님")

    if not all(counts[f"chr{i}"] == 2 for i in range(1, 23)):
        reasons.append("1~22번이 각각 2개인 구성과 다름")

    if (counts["chrX"], counts["chrY"]) not in [(2, 0), (1, 1)]:
        reasons.append("XX 또는 XY 주석 구성과 다름")

    if unknown_labels:
        reasons.append("알 수 없는 번호 라벨")
    if invalid_boxes:
        reasons.append("면적이 잘못된 상자")
    if outside_boxes:
        reasons.append("사진 경계를 벗어난 상자")

    audit_records.append({
        **case.to_dict(),
        "annotation_check_pass": not reasons,
        "review_reason": " / ".join(reasons),
    })

additional_1000_audit = pd.DataFrame(audit_records)

additional_training_candidates = additional_1000_audit.loc[
    additional_1000_audit["annotation_check_pass"]
].copy()

additional_needs_review = additional_1000_audit.loc[
    ~additional_1000_audit["annotation_check_pass"]
].copy()

# 기존 학습·검증·테스트 사진과 중복되지 않는지 확인
existing_names = (
    set(expanded_train_cases["filename"])
    | set(expanded_validation_cases["filename"])
    | set(expanded_test_cases["filename"])
)

assert not (
    set(additional_training_candidates["filename"])
    & existing_names
)

additional_1000_audit.to_csv(
    BASE / "normal_additional_1000_audit_v1.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n검사한 사진:", len(acquired_cases), "장")
print("추가 학습 후보:", len(additional_training_candidates), "장")
print("별도 검토 필요:", len(additional_needs_review), "장")
print(
    "통과 자료를 추가했을 때 학습 원본:",
    len(expanded_train_cases) + len(additional_training_candidates),
    "장",
)

display(
    additional_needs_review[
        ["filename", "xml_object_count", "review_reason"]
    ]
)

print("주석 검사 완료 — 아직 학습 자료에 합치거나 재학습하지 않았어.")

In [ ]:
import pandas as pd

# 1. 새 학습 원본 목록
new481_cases = (
    additional_training_candidates
    .sort_values("filename")
    .reset_index(drop=True)
    .copy()
)
new481_cases["data_cohort"] = "additional_normal_cases"

training652_cases = pd.concat(
    [expanded_train_cases, new481_cases],
    ignore_index=True,
    sort=False,
)

assert training652_cases["filename"].is_unique

held_out_names = (
    set(expanded_validation_cases["filename"])
    | set(expanded_test_cases["filename"])
)
assert not (set(training652_cases["filename"]) & held_out_names)

training652_split_dir = BASE / "splits" / "tvg_number_652_v1"
training652_split_dir.mkdir(parents=True, exist_ok=True)

for name, table in [
    ("train", training652_cases),
    ("validation", expanded_validation_cases),
    ("test", expanded_test_cases),
]:
    table.to_csv(
        training652_split_dir / f"{name}_cases.csv",
        index=False,
        encoding="utf-8-sig",
    )

# 2. 新規481枚だけ、以前と同じ方法で切り出す
split_dir = training652_split_dir
output_root = BASE / "derived" / "number_shared_652_v1"

new481_crops = extract_number_crops(
    new481_cases, "train_additional481_v1"
)

new481_crops = new481_crops.merge(
    new481_cases[["filename", "data_cohort"]],
    left_on="source_image",
    right_on="filename",
    how="left",
    validate="many_to_one",
).drop(columns=["filename"])

assert new481_crops["data_cohort"].notna().all()
assert len(new481_crops) == len(new481_cases) * 46

# 3. 同じ背景処理と拡大画面作成
new481_masked_crops = save_masked_crops(
    new481_crops, "train_additional481_v1"
)

new481_dual_views = create_local_views(
    new481_masked_crops, "train_additional481_v1"
)

# 4. 既存の学習用画面一覧と結合
training652_dual_views = pd.concat(
    [dual_train_views, new481_dual_views],
    ignore_index=True,
    sort=False,
)

assert not training652_dual_views.duplicated(
    ["source_image", "object_index"]
).any()

assert (
    training652_dual_views["source_image"].nunique()
    == len(training652_cases)
)

training652_dual_views.to_csv(
    training652_split_dir / "train_views.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n===== 拡張入力準備完了 =====")
print("학습 원본:", len(training652_cases), "장")
print("새로 추가한 조각:", len(new481_dual_views), "개")
print("전체 학습 조각:", len(training652_dual_views), "개")
print("검증 원본:", len(expanded_validation_cases), "장 — 유지")
print("테스트 원본:", len(expanded_test_cases), "장 — 유지")
print("목록 저장 위치:", training652_split_dir)
print("아직 새 자료의 특징 추출·학습 전")

In [ ]:
import torch

training652_feature_dir = BASE / "features" / "number_652_dual_v1"
training652_feature_dir.mkdir(parents=True, exist_ok=True)

# 기존 추출 함수의 저장 경로를 이번 실험 폴더로 설정
masked_feature_dir = training652_feature_dir / "global"
dual_feature_dir = training652_feature_dir / "dual"

masked_feature_dir.mkdir(parents=True, exist_ok=True)
dual_feature_dir.mkdir(parents=True, exist_ok=True)

split_name = "train_additional481_v1"

# 1. 새 자료의 기존 화면 특징
global_path = masked_feature_dir / f"{split_name}_features.pt"

if global_path.is_file():
    new481_global_features = torch.load(
        global_path,
        map_location="cpu",
        weights_only=True,
        mmap=True,
    )
    assert (
        new481_global_features["crop_paths"]
        == new481_masked_crops["crop_path"].tolist()
    )
    assert new481_global_features["features"].shape == (22126, 1024)
    assert torch.equal(
        new481_global_features["labels"],
        torch.tensor(
            new481_masked_crops["target"].to_numpy(),
            dtype=torch.long,
        ),
    )
    assert (
        new481_global_features["metadata"]["preprocessing"]
        == str(tvg_transform)
    )
    assert torch.isfinite(new481_global_features["features"]).all().item()
    print("새 자료의 기존 화면 특징 재사용")
else:
    new481_global_features = extract_masked_features(
        new481_masked_crops, split_name
    )

# 2. 확대 특징 추출 후 기존 화면 특징과 연결
dual_path = dual_feature_dir / f"{split_name}_features.pt"

if dual_path.is_file():
    new481_dual_features = torch.load(
        dual_path,
        map_location="cpu",
        weights_only=True,
        mmap=True,
    )
    assert (
        new481_dual_features["global_crop_paths"]
        == new481_dual_views["global_crop_path"].tolist()
    )
    assert (
        new481_dual_features["local_crop_paths"]
        == new481_dual_views["local_crop_path"].tolist()
    )
    assert new481_dual_features["features"].shape == (22126, 2048)
    assert torch.equal(
        new481_dual_features["labels"],
        new481_global_features["labels"],
    )
    assert torch.equal(
        new481_dual_features["features"][:, :1024],
        new481_global_features["features"],
    )
    assert torch.isfinite(new481_dual_features["features"]).all().item()
    print("새 자료의 통합 특징 재사용")
else:
    new481_dual_features = extract_dual_features(
        new481_dual_views,
        new481_global_features,
        split_name,
    )

# 3. 기존 171장 특징과 새 481장 특징 합치기
for key in [
    "encoder_checkpoint",
    "preprocessing",
    "background_method",
    "local_view_method",
    "feature_order",
]:
    assert (
        dual_train_features["metadata"][key]
        == new481_dual_features["metadata"][key]
    )

training652_features = {
    "features": torch.cat([
        dual_train_features["features"],
        new481_dual_features["features"],
    ], dim=0),
    "labels": torch.cat([
        dual_train_features["labels"],
        new481_dual_features["labels"],
    ], dim=0),
    "chromosome_names": number_names,
    "metadata": {
        **dual_train_features["metadata"],
        "experiment": "training_652_fixed_encoder_v1",
        "split_directory": str(training652_split_dir),
        "training_source_images": 652,
    },
}

for key in [
    "global_crop_paths", "local_crop_paths",
    "source_images", "object_indices", "data_cohorts",
]:
    training652_features[key] = (
        dual_train_features[key] + new481_dual_features[key]
    )

# 特徴と写真一覧の並びを確認
assert training652_features["features"].shape == (29992, 2048)
assert (
    training652_features["global_crop_paths"]
    == training652_dual_views["global_crop_path"].tolist()
)
assert (
    training652_features["local_crop_paths"]
    == training652_dual_views["local_crop_path"].tolist()
)
assert torch.equal(
    training652_features["labels"],
    torch.tensor(
        training652_dual_views["target"].to_numpy(),
        dtype=torch.long,
    ),
)
assert torch.isfinite(training652_features["features"]).all().item()

destination = training652_feature_dir / "train_features.pt"
temporary = destination.with_suffix(".pt.part")
torch.save(training652_features, temporary)
temporary.replace(destination)

print("\n===== 特徴準備完了 =====")
print("학습 특징:", tuple(training652_features["features"].shape))
print("검증 특징:", tuple(dual_validation_features["features"].shape))
print("저장 위치:", destination)
print("검증 특징은 기존 자료 그대로 사용해.")
print("아직 652장 자료를 이용한 분류기 학습 전")

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# 1. 학습·검증 특징 준비
X_train_652 = training652_features["features"].numpy()
y_train_652 = training652_features["labels"].numpy()

X_val_fixed = dual_validation_features["features"].numpy()
y_val_fixed = dual_validation_features["labels"].numpy()

# 인코더와 입력 생성 방식이 같은지 확인
for key in [
    "encoder_checkpoint",
    "preprocessing",
    "background_method",
    "local_view_method",
    "feature_order",
]:
    assert (
        training652_features["metadata"][key]
        == dual_validation_features["metadata"][key]
    )

assert set(y_train_652.tolist()) == set(range(24))
assert X_train_652.shape == (29992, 2048)

# 2. 기존과 동일한 설정으로 새 분류기 학습
classifier_652 = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.1,
        max_iter=3000,
        solver="lbfgs",
        random_state=42,
    ),
)

print("652장 분류기 학습 시작 — 진행 막대 없이 실행돼.")
classifier_652.fit(X_train_652, y_train_652)

train_pred_652 = classifier_652.predict(X_train_652)
val_pred_652 = classifier_652.predict(X_val_fixed)

print(
    f"\n학습 정확도: "
    f"{accuracy_score(y_train_652, train_pred_652):.2%}"
)

# 3. 같은 검증 57장에서 기존 모델과 비교
cohort_array = np.array(
    dual_validation_features["data_cohorts"]
)

groups = {
    "전체 57장": np.ones(len(y_val_fixed), dtype=bool),
    "기존 20장": cohort_array == "original_abnormal_cases",
    "추가 정상 37장": cohort_array == "additional_normal_cases",
}

comparison_rows = []

for model_name, classifier in [
    ("학습 원본 171장", dual_reg_classifiers[0.1]),
    ("학습 원본 652장", classifier_652),
]:
    predictions = classifier.predict(X_val_fixed)

    for group_name, mask in groups.items():
        actual = y_val_fixed[mask]
        predicted = predictions[mask]

        actual_21 = actual == number_to_id["chr21"]
        predicted_21 = predicted == number_to_id["chr21"]

        photos = pd.DataFrame({
            "source_image": np.array(
                dual_validation_features["source_images"]
            )[mask],
            "actual": actual_21.astype(int),
            "predicted": predicted_21.astype(int),
            "fp": (~actual_21 & predicted_21).astype(int),
            "fn": (actual_21 & ~predicted_21).astype(int),
        }).groupby("source_image").sum()

        comparison_rows.append({
            "모델": model_name,
            "검증 자료": group_name,
            "번호 정확도": accuracy_score(actual, predicted),
            "Macro F1": f1_score(
                actual, predicted,
                labels=list(range(24)),
                average="macro",
                zero_division=0,
            ),
            "21번 개수 일치 사진": int(
                photos["actual"].eq(photos["predicted"]).sum()
            ),
            "21번 식별 모두 정확한 사진": int(
                (photos["fp"].eq(0) & photos["fn"].eq(0)).sum()
            ),
            "21번 오분류": int(photos["fp"].sum()),
            "21번 놓침": int(photos["fn"].sum()),
        })

comparison_652 = pd.DataFrame(comparison_rows)
display(comparison_652.round(4))

# 4. 모델과 결과를 별도 저장
model_652_dir = BASE / "models" / "number_652_dual_linear_v1"
model_652_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "classifier": classifier_652,
    "chromosome_names": number_names,
    "class_to_idx": number_to_id,
    "feature_metadata": training652_features["metadata"],
    "C": 0.1,
    "training_accuracy": float(
        accuracy_score(y_train_652, train_pred_652)
    ),
}, model_652_dir / "classifier.joblib")

comparison_652.to_csv(
    model_652_dir / "validation_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

validation_results_652 = dual_validation_views.copy()
validation_results_652["predicted_chromosome"] = [
    number_names[int(i)] for i in val_pred_652
]
validation_results_652.to_csv(
    model_652_dir / "validation_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n학습·비교·저장 완료:", model_652_dir)
print("테스트 자료는 이번 실험에 사용하지 않았어.")

In [ ]:
import copy
import numpy as np
import torch
from PIL import Image

# 이전 실험 모델은 유지하고 별도 이름으로 준비
if "dual652_finetune_model" in globals():
    raise RuntimeError(
        "dual652_finetune_model이 이미 있어. "
        "이 셀을 다시 실행하지 말고 현재 상태를 알려줘."
    )

# 미세조정 전 인코더를 복사
dual652_finetune_model = TVGDualFineTune(
    copy.deepcopy(tvg_number_encoder)
).to(device)

# 652장 분류기의 표준화와 분류층을 하나의 선형층으로 변환
scaler652 = classifier_652.named_steps["standardscaler"]
linear652 = classifier_652.named_steps["logisticregression"]

assert np.array_equal(linear652.classes_, np.arange(24))

weight652 = linear652.coef_ / scaler652.scale_[None, :]
bias652 = linear652.intercept_ - weight652 @ scaler652.mean_

with torch.no_grad():
    dual652_finetune_model.head.weight.copy_(
        torch.tensor(weight652, dtype=torch.float32, device=device)
    )
    dual652_finetune_model.head.bias.copy_(
        torch.tensor(bias652, dtype=torch.float32, device=device)
    )

dual652_finetune_model.requires_grad_(False)

for module in [
    dual652_finetune_model.encoder.blocks[-2],
    dual652_finetune_model.encoder.blocks[-1],
    dual652_finetune_model.encoder.fc_norm,
    dual652_finetune_model.head,
]:
    module.requires_grad_(True)

# 同じ入力で出発予測が一致するか確認
global_images = []
local_images = []

for _, record in dual_validation_views.iloc[:2].iterrows():
    with Image.open(record["global_crop_path"]) as source:
        global_images.append(tvg_transform(source.convert("RGB")))

    with Image.open(record["local_crop_path"]) as source:
        local_images.append(tvg_transform(source.convert("RGB")))

dual652_finetune_model.eval()

with torch.inference_mode():
    new_predictions = dual652_finetune_model(
        torch.stack(global_images).to(device),
        torch.stack(local_images).to(device),
    ).argmax(1).cpu().numpy()

old_predictions = classifier_652.predict(X_val_fixed[:2])
assert np.array_equal(new_predictions, old_predictions)

dual652_finetune_dir = (
    BASE / "models" / "number_652_dual_finetune_v1"
)
dual652_finetune_dir.mkdir(parents=True, exist_ok=True)

trainable = sum(
    p.numel() for p in dual652_finetune_model.parameters()
    if p.requires_grad
)

print("出発モデル: 学習原本652枚の分類器")
print("출발 검증 정확도: 85.49%")
print("시험 입력의 기존 예측과 일치: True")
print(f"학습 가능한 파라미터: {trainable:,}개")
print("학습 원본: 652장")
print("검증 원본: 기존 57장 유지")
print("저장 예정 위치:", dual652_finetune_dir)
print("미세조정 준비 완료 — 아직 학습 전")

In [ ]:
import random
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

history_path652 = dual652_finetune_dir / "history.csv"

if history_path652.exists():
    raise RuntimeError(
        "이 폴더에 학습 기록이 이미 있어. "
        "다시 실행하지 말고 현재 상태를 알려줘."
    )

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# 앞서 만든 두 화면 데이터셋 사용
train_loader652 = DataLoader(
    DualViewDataset(training652_dual_views, augment=True),
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader652 = DataLoader(
    DualViewDataset(dual_validation_views, augment=False),
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

criterion652 = nn.CrossEntropyLoss()

optimizer652 = torch.optim.AdamW([
    {
        "params": [
            p for p in dual652_finetune_model.encoder.parameters()
            if p.requires_grad
        ],
        "lr": 2e-6,
    },
    {
        "params": dual652_finetune_model.head.parameters(),
        "lr": 1e-5,
    },
], weight_decay=1e-4)

trainable652 = [
    p for p in dual652_finetune_model.parameters()
    if p.requires_grad
]


def evaluate652():
    dual652_finetune_model.eval()
    actual, predicted = [], []
    loss_sum = 0.0

    with torch.inference_mode():
        for global_images, local_images, labels in tqdm(
            val_loader652, desc="검증"
        ):
            labels = labels.to(device)

            logits = dual652_finetune_model(
                global_images.to(device),
                local_images.to(device),
            )

            loss_sum += (
                criterion652(logits, labels).item() * len(labels)
            )
            actual.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(1).cpu().tolist())

    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    assert np.array_equal(
        actual, dual_validation_views["target"].to_numpy()
    )

    actual21 = actual == number_to_id["chr21"]
    predicted21 = predicted == number_to_id["chr21"]

    photos = pd.DataFrame({
        "사진": dual_validation_views["source_image"].to_numpy(),
        "실제": actual21.astype(int),
        "예측": predicted21.astype(int),
        "오분류": (~actual21 & predicted21).astype(int),
        "놓침": (actual21 & ~predicted21).astype(int),
    }).groupby("사진").sum()

    return {
        "loss": float(loss_sum / len(actual)),
        "accuracy": float(accuracy_score(actual, predicted)),
        "macro_f1": float(f1_score(
            actual, predicted,
            labels=list(range(24)),
            average="macro",
            zero_division=0,
        )),
        "chr21_count_correct": int(
            photos["실제"].eq(photos["예측"]).sum()
        ),
        "chr21_identification_correct": int(
            (photos["오분류"].eq(0) & photos["놓침"].eq(0)).sum()
        ),
        "chr21_false_positive": int(photos["오분류"].sum()),
        "chr21_false_negative": int(photos["놓침"].sum()),
    }


def save_best652(epoch, metrics):
    destination = dual652_finetune_dir / "best.pth"
    temporary = destination.with_suffix(".pth.part")

    torch.save({
        "model": {
            key: value.detach().cpu()
            for key, value in dual652_finetune_model.state_dict().items()
        },
        "epoch": epoch,
        "validation_metrics": metrics,
        "architecture": "shared_vit_large_patch16_dual_view",
        "global_pool": "avg",
        "chromosome_names": number_names,
        "feature_metadata": training652_features["metadata"],
        "training_augmentation": (
            "synchronized_0_90_180_270_rotation_and_horizontal_flip"
        ),
        "selection_metric": "validation_macro_f1",
    }, temporary)

    temporary.replace(destination)


# 출발 성능과 가중치 보존
baseline652 = evaluate652()
best_f1652 = baseline652["macro_f1"]
best_epoch652 = 0
no_improvement652 = 0

save_best652(0, baseline652)

history652 = [{
    "epoch": 0,
    "train_loss": None,
    "train_accuracy": None,
    **{f"validation_{k}": v for k, v in baseline652.items()},
}]

pd.DataFrame(history652).to_csv(
    history_path652, index=False, encoding="utf-8-sig"
)

print(
    f"\n출발 성능 | 검증 정확도 {baseline652['accuracy']:.2%}"
    f" | Macro F1 {baseline652['macro_f1']:.4f}"
    f" | 21번 개수 일치 {baseline652['chr21_count_correct']}/57장"
)

# 배치 2개씩 8회 누적하여 16개 기준으로 가중치 갱신
accumulation_steps652 = 8

for epoch in range(1, 7):
    # 고정된 앞부분은 평가 모드 유지
    dual652_finetune_model.eval()

    for block in dual652_finetune_model.encoder.blocks[-2:]:
        block.train()

    dual652_finetune_model.encoder.fc_norm.train()
    dual652_finetune_model.head.train()

    optimizer652.zero_grad(set_to_none=True)

    loss_sum = 0.0
    correct = 0
    total = 0
    loader_length = len(train_loader652)

    for batch_index, (global_images, local_images, labels) in enumerate(
        tqdm(train_loader652, desc=f"학습 {epoch}/6")
    ):
        labels = labels.to(device)

        logits = dual652_finetune_model(
            global_images.to(device),
            local_images.to(device),
        )
        loss = criterion652(logits, labels)

        if not torch.isfinite(loss).item():
            raise RuntimeError("손실값이 비정상이어서 학습을 중단했어.")

        group_start = (
            batch_index // accumulation_steps652
        ) * accumulation_steps652

        group_size = min(
            accumulation_steps652, loader_length - group_start
        )

        (loss / group_size).backward()

        if (
            (batch_index + 1) % accumulation_steps652 == 0
            or batch_index + 1 == loader_length
        ):
            torch.nn.utils.clip_grad_norm_(
                trainable652, max_norm=1.0
            )
            optimizer652.step()
            optimizer652.zero_grad(set_to_none=True)

        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        total += len(labels)

    metrics = evaluate652()
    improved = metrics["macro_f1"] > best_f1652 + 1e-6

    if improved:
        best_f1652 = metrics["macro_f1"]
        best_epoch652 = epoch
        no_improvement652 = 0
        save_best652(epoch, metrics)
    else:
        no_improvement652 += 1

    history652.append({
        "epoch": epoch,
        "train_loss": loss_sum / total,
        "train_accuracy": correct / total,
        **{f"validation_{k}": v for k, v in metrics.items()},
    })

    pd.DataFrame(history652).to_csv(
        history_path652, index=False, encoding="utf-8-sig"
    )

    print(
        f"\n에포크 {epoch}/6"
        f" | 증강 학습 정확도 {correct / total:.2%}"
        f" | 검증 정확도 {metrics['accuracy']:.2%}"
        f" | 검증 Macro F1 {metrics['macro_f1']:.4f}"
        f" | 21번 개수 일치 {metrics['chr21_count_correct']}/57장"
        f" | 오분류 {metrics['chr21_false_positive']}개"
        f" / 놓침 {metrics['chr21_false_negative']}개"
        + (" ★ 최적 모델 저장" if improved else "")
    )

    if no_improvement652 >= 3:
        print("검증 Macro F1이 3에포크 연속 개선되지 않아 중단했어.")
        break

dual652_finetune_model.eval()

print("\n학습 완료")
print("최적 에포크:", best_epoch652, "(0이면 출발 모델)")
print(f"최적 검증 Macro F1: {best_f1652:.4f}")
print("저장 위치:", dual652_finetune_dir)
print("현재 메모리에는 마지막 에포크 모델이 있어.")
print("다음 평가는 저장된 최적 모델을 복원해서 진행할 거야.")

In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

# 1. 최적 모델 복원
best652 = torch.load(
    dual652_finetune_dir / "best.pth",
    map_location="cpu",
    weights_only=True,
    mmap=True,
)

dual652_finetune_model.load_state_dict(
    best652["model"], strict=True
)
dual652_finetune_model.eval()

print("복원한 에포크:", best652["epoch"])

# 2. 검증 예측
actual652 = []
predicted652 = []

with torch.inference_mode():
    for global_images, local_images, labels in tqdm(
        val_loader652, desc="최적 모델 오류 분석"
    ):
        logits = dual652_finetune_model(
            global_images.to(device),
            local_images.to(device),
        )
        actual652.extend(labels.tolist())
        predicted652.extend(logits.argmax(1).cpu().tolist())

review652 = dual_validation_views.copy().reset_index(drop=True)

assert actual652 == review652["target"].tolist()

review652["predicted_chromosome"] = [
    number_names[i] for i in predicted652
]
review652["correct"] = (
    review652["chromosome"] == review652["predicted_chromosome"]
)

# 3. 번호별 성능
report652 = classification_report(
    actual652,
    predicted652,
    labels=list(range(24)),
    target_names=number_names,
    output_dict=True,
    zero_division=0,
)

number_report652 = pd.DataFrame(report652).T.loc[
    number_names, ["precision", "recall", "f1-score", "support"]
]

number_report652.columns = [
    "정밀도", "재현율", "F1", "검증 조각 수"
]

print("\n재현율이 낮은 번호 8개:")
display(
    number_report652.sort_values("재현율").head(8).round(4)
)

# 4. 자주 혼동한 번호 조합
errors652 = review652.loc[~review652["correct"]]

confusions652 = (
    errors652.groupby(
        ["chromosome", "predicted_chromosome"]
    )
    .size()
    .reset_index(name="오류 수")
    .sort_values("오류 수", ascending=False)
)

print("\n자주 혼동한 번호 조합 10개:")
display(confusions652.head(10))

# 5. 사진별 21번 개수와 식별 평가
photo_records652 = []

for filename, group in review652.groupby("source_image"):
    actual = group["chromosome"].eq("chr21")
    predicted = group["predicted_chromosome"].eq("chr21")

    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())

    photo_records652.append({
        "source_image": filename,
        "actual_count": int(actual.sum()),
        "predicted_count": int(predicted.sum()),
        "false_positive": fp,
        "false_negative": fn,
        "count_correct": int(actual.sum()) == int(predicted.sum()),
        "identification_correct": fp == 0 and fn == 0,
    })

photo_review652 = pd.DataFrame(photo_records652)

print(
    "\n21번 개수 일치:",
    f"{photo_review652['count_correct'].sum()}/57장",
)
print(
    "21번을 오분류·놓침 없이 구분:",
    f"{photo_review652['identification_correct'].sum()}/57장",
)

# 6. 분석 기록 저장
review652.to_csv(
    dual652_finetune_dir / "validation_predictions.csv",
    index=False, encoding="utf-8-sig",
)
number_report652.to_csv(
    dual652_finetune_dir / "validation_per_chromosome.csv",
    encoding="utf-8-sig",
)
confusions652.to_csv(
    dual652_finetune_dir / "validation_confusions.csv",
    index=False, encoding="utf-8-sig",
)
photo_review652.to_csv(
    dual652_finetune_dir / "validation_chr21_review.csv",
    index=False, encoding="utf-8-sig",
)

print("최적 모델 복원·오류 분석 완료")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# 1. 성능이 낮은 번호의 학습 자료 수 확인
train_counts = training652_dual_views.groupby("chromosome").size()

data_review = number_report652.copy()
data_review["학습 조각 수"] = (
    data_review.index.to_series().map(train_counts).to_numpy()
)

print("재현율이 낮은 번호의 데이터 수:")
display(
    data_review.sort_values("재현율")
    .head(8)[["학습 조각 수", "검증 조각 수", "정밀도", "재현율"]]
    .round(4)
)

# 2. 주요 혼동에서 서로 다른 원본 사진의 예시 선택
conditions = [
    ("4번 → 5번", (
        review652["chromosome"].eq("chr4")
        & review652["predicted_chromosome"].eq("chr5")
    )),
    ("22번 → 19번", (
        review652["chromosome"].eq("chr22")
        & review652["predicted_chromosome"].eq("chr19")
    )),
    ("X → 8번", (
        review652["chromosome"].eq("chrX")
        & review652["predicted_chromosome"].eq("chr8")
    )),
    ("Y 놓침", (
        review652["chromosome"].eq("chrY")
        & ~review652["predicted_chromosome"].eq("chrY")
    )),
]

examples = []

for title, condition in conditions:
    candidates = (
        review652.loc[condition]
        .sort_values(["source_image", "object_index"])
        .drop_duplicates("source_image")
    )

    selected = candidates.sample(
        n=min(2, len(candidates)),
        random_state=42,
    )

    for _, record in selected.iterrows():
        examples.append((title, record))

fig, axes = plt.subplots(
    len(examples), 3,
    figsize=(12, 3 * len(examples)),
    squeeze=False,
)

for row_index, (title, record) in enumerate(examples):
    paths = [
        record["original_crop_path"],
        record["global_crop_path"],
        record["local_crop_path"],
    ]

    for column, path in enumerate(paths):
        with Image.open(path) as source:
            axes[row_index, column].imshow(source.convert("RGB"))
        axes[row_index, column].axis("off")

    axes[row_index, 0].set_title(
        f"Original: {record['chromosome']}\n"
        f"{record['source_image']}",
        fontsize=9,
    )
    axes[row_index, 1].set_title("Masked global")
    axes[row_index, 2].set_title(
        f"Local / Pred: {record['predicted_chromosome']}"
    )

plt.tight_layout()
plt.show()

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

quality_records = []

for _, case in expanded_validation_cases.iterrows():
    root = ET.parse(BASE / case["annotation_path"]).getroot()
    width = int(root.findtext("size/width"))
    height = int(root.findtext("size/height"))

    boxes = []

    for obj in root.findall("object"):
        x1, y1, x2, y2 = [
            int(obj.findtext(f"bndbox/{key}"))
            for key in ["xmin", "ymin", "xmax", "ymax"]
        ]
        boxes.append((
            max(0, x1), max(0, y1),
            min(width, x2), min(height, y2),
        ))

    for index, (x1, y1, x2, y2) in enumerate(boxes):
        box_width = x2 - x1
        box_height = y2 - y1
        area = box_width * box_height
        assert area > 0

        max_overlap = 0.0

        for other_index, (ox1, oy1, ox2, oy2) in enumerate(boxes):
            if index == other_index:
                continue

            intersection = (
                max(0, min(x2, ox2) - max(x1, ox1))
                * max(0, min(y2, oy2) - max(y1, oy1))
            )
            max_overlap = max(max_overlap, intersection / area)

        quality_records.append({
            "source_image": case["filename"],
            "object_index": index,
            "원본_긴변_픽셀": max(box_width, box_height),
            "최대_상자_겹침": max_overlap,
        })

quality_review652 = review652.merge(
    pd.DataFrame(quality_records),
    on=["source_image", "object_index"],
    how="left",
    validate="one_to_one",
)

assert quality_review652["원본_긴변_픽셀"].notna().all()

# 원본 크기를 작은 순서대로 네 구간으로 나눔
quality_review652["원본 크기 구간"] = pd.qcut(
    quality_review652["원본_긴변_픽셀"],
    q=4,
    duplicates="drop",
)

quality_review652["상자 겹침 구분"] = (
    quality_review652["최대_상자_겹침"]
    .ge(0.2)
    .map({False: "20% 미만", True: "20% 이상"})
)

for column in ["원본 크기 구간", "상자 겹침 구분"]:
    print(f"\n{column}별 성능:")
    display(
        quality_review652.groupby(column, observed=True)
        .agg(
            조각_수=("correct", "size"),
            오류_수=("correct", lambda values: int((~values).sum())),
            정확도=("correct", "mean"),
        )
        .round(4)
    )

quality_review652.to_csv(
    dual652_finetune_dir / "validation_input_quality.csv",
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
import sys
import importlib.util
import torch

print("파이썬:", sys.executable)

for module in ["segment_anything", "cv2"]:
    installed = importlib.util.find_spec(module) is not None
    print(f"{module}:", "설치됨" if installed else "미설치")

print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"전체 GPU 메모리: {total_bytes / 1024**3:.2f} GiB")
    print(f"현재 여유 GPU 메모리: {free_bytes / 1024**3:.2f} GiB")

print("환경 확인 완료 — 설치·다운로드·학습 없음")

In [ ]:
import numpy as np
import torch
from packaging.version import Version

if Version(np.__version__) < Version("1.26.0"):
    raise RuntimeError(
        f"현재 NumPy는 {np.__version__}이야. "
        "설치를 진행하지 말고 이 메시지를 보내줘."
    )

torch_version_before = torch.__version__
numpy_version_before = np.__version__

# 기존 의존성은 변경하지 않고 필요한 패키지만 설치
%pip install --no-deps "opencv-python-headless==4.11.0.86"
%pip install --no-deps "https://github.com/facebookresearch/segment-anything/archive/dca509fe793f601edb92606367a655c15ac00fdf.zip"

# 설치 후 가져오기 확인
import cv2
import segment_anything
from segment_anything import SamPredictor, sam_model_registry

assert torch.__version__ == torch_version_before
assert np.__version__ == numpy_version_before

print("\nOpenCV:", cv2.__version__)
print("SAM 가져오기: 성공")
print("ViT-B 모델 등록:", "vit_b" in sam_model_registry)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
print("패키지 준비 완료 — 가중치 다운로드·학습 전")

In [ ]:
from pathlib import Path
import urllib.request
import hashlib
import torch
from tqdm.auto import tqdm

sam_dir = Path(r"C:\chromosome_ai\pretrained\SAM")
sam_dir.mkdir(parents=True, exist_ok=True)

sam_checkpoint_path = sam_dir / "sam_vit_b_01ec64.pth"
sam_url = (
    "https://dl.fbaipublicfiles.com/segment_anything/"
    "sam_vit_b_01ec64.pth"
)

if sam_checkpoint_path.is_file():
    print("기존 가중치 파일 확인 중")
else:
    temporary = sam_checkpoint_path.with_suffix(".pth.part")

    print("SAM ViT-B 가중치 다운로드 시작")

    with urllib.request.urlopen(sam_url, timeout=60) as response:
        total = int(response.headers.get("Content-Length", 0))
        downloaded = 0

        with temporary.open("wb") as output, tqdm(
            total=total or None,
            unit="B",
            unit_scale=True,
            desc="SAM 다운로드",
        ) as progress:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break

                output.write(chunk)
                downloaded += len(chunk)
                progress.update(len(chunk))

    if total and downloaded != total:
        raise RuntimeError(
            "서버가 안내한 크기와 다운로드 크기가 달라. "
            "이 오류를 알려줘."
        )

    temporary.replace(sam_checkpoint_path)

# 안전한 가중치 읽기로 파일 형식·구조 확인
sam_state_check = torch.load(
    sam_checkpoint_path,
    map_location="cpu",
    weights_only=True,
    mmap=True,
)

assert sam_state_check[
    "image_encoder.patch_embed.proj.weight"
].shape == (768, 3, 16, 16)

assert sam_state_check[
    "image_encoder.pos_embed"
].shape == (1, 64, 64, 768)

# 이후 같은 파일인지 확인할 수 있도록 해시 기록
sha256 = hashlib.sha256()

with sam_checkpoint_path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        sha256.update(chunk)

hash_value = sha256.hexdigest()
sam_checkpoint_path.with_suffix(".sha256").write_text(
    hash_value + "\n",
    encoding="utf-8",
)

print("\n파일 읽기·ViT-B 주요 구조 확인: 정상")
print("저장 위치:", sam_checkpoint_path)
print(f"파일 크기: {sam_checkpoint_path.stat().st_size / 1024**2:.1f} MiB")
print("가중치 항목 수:", len(sam_state_check))
print("SHA-256:", hash_value)
print("가중치 준비 완료 — 아직 GPU 로딩·분리 시험 전")

del sam_state_check

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from PIL import Image
from matplotlib.patches import Rectangle
from segment_anything import SamPredictor, sam_model_registry

# 1. SAM을 안전하게 불러오기
if "sam_predictor" not in globals():
    sam_model = sam_model_registry["vit_b"](checkpoint=None)

    sam_weights = torch.load(
        sam_checkpoint_path,
        map_location="cpu",
        weights_only=True,
        mmap=True,
    )
    loading_result = sam_model.load_state_dict(
        sam_weights, strict=True
    )
    del sam_weights

    sam_model.requires_grad_(False)
    sam_model.eval()
    sam_model.to(device)

    sam_predictor = SamPredictor(sam_model)
    print("SAM 가중치 로딩:", loading_result)

# 2. 학습 자료에서 상자가 겹치는 4번 염색체 사례 선택
sam_trial_case = None

for _, case in (
    training652_cases.sort_values("filename")
    .sample(frac=1, random_state=42)
    .iterrows()
):
    root = ET.parse(BASE / case["annotation_path"]).getroot()
    objects = root.findall("object")

    width = int(root.findtext("size/width"))
    height = int(root.findtext("size/height"))

    boxes = []

    for obj in objects:
        x1, y1, x2, y2 = [
            int(obj.findtext(f"bndbox/{key}"))
            for key in ["xmin", "ymin", "xmax", "ymax"]
        ]
        boxes.append((
            max(0, x1), max(0, y1),
            min(width, x2), min(height, y2),
        ))

    for index, obj in enumerate(objects):
        if (obj.findtext("name") or "").strip() != "B4":
            continue

        x1, y1, x2, y2 = boxes[index]
        area = (x2 - x1) * (y2 - y1)
        if area <= 0:
            continue

        overlap = 0.0

        for other_index, (ox1, oy1, ox2, oy2) in enumerate(boxes):
            if index == other_index:
                continue

            intersection = (
                max(0, min(x2, ox2) - max(x1, ox1))
                * max(0, min(y2, oy2) - max(y1, oy1))
            )
            overlap = max(overlap, intersection / area)

        if overlap >= 0.2:
            sam_trial_case = case.copy()
            sam_trial_box = np.array(boxes[index], dtype=np.float32)
            sam_trial_object_index = index
            sam_trial_overlap = overlap
            break

    if sam_trial_case is not None:
        break

if sam_trial_case is None:
    raise RuntimeError("조건에 맞는 학습 사례를 찾지 못했어.")

# 3. 전체 원본 사진과 대상 상자로 마스크 후보 생성
with Image.open(BASE / sam_trial_case["image_path"]) as source:
    sam_trial_image = np.asarray(source.convert("RGB"))

with torch.inference_mode():
    sam_predictor.set_image(sam_trial_image)

    sam_trial_masks, sam_trial_scores, _ = sam_predictor.predict(
        box=sam_trial_box,
        multimask_output=True,
    )

# 4. 같은 영역에서 원본과 후보 3개 비교
x1, y1, x2, y2 = sam_trial_box.astype(int)
margin = 15

left = max(0, x1 - margin)
top = max(0, y1 - margin)
right = min(sam_trial_image.shape[1], x2 + margin)
bottom = min(sam_trial_image.shape[0], y2 + margin)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))

axes[0].imshow(sam_trial_image[top:bottom, left:right])
axes[0].add_patch(Rectangle(
    (x1 - left, y1 - top),
    x2 - x1,
    y2 - y1,
    fill=False,
    edgecolor="red",
    linewidth=1.5,
))
axes[0].set_title("원본과 대상 상자")

for index, (mask, score) in enumerate(
    zip(sam_trial_masks, sam_trial_scores)
):
    cleaned = sam_trial_image.copy()
    cleaned[~mask] = 255

    axes[index + 1].imshow(
        cleaned[top:bottom, left:right]
    )
    axes[index + 1].set_title(
        f"후보 {index + 1}\nSAM 점수 {float(score):.3f}"
    )

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("학습 원본:", sam_trial_case["filename"])
print("대상: XML에 4번으로 주석된 염색체")
print("객체 인덱스:", sam_trial_object_index)
print(f"최대 상자 겹침 비율: {sam_trial_overlap:.1%}")
print("상자는 정답 주석에서 제공했어. 자동 검출 시험은 아니야.")
print("분리 후보 확인 완료 — 파일 변경·재학습 없음")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text
from IPython.display import display

font_path = Path(r"C:\Windows\Fonts\malgun.ttf")

if not font_path.is_file():
    raise FileNotFoundError("맑은 고딕 폰트를 찾지 못했어.")

font_manager.fontManager.addfont(str(font_path))
font_name = font_manager.FontProperties(
    fname=str(font_path)
).get_name()

# 앞으로 만드는 그래프에도 적용
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 이미 만든 SAM 그래프에도 적용
for text in fig.findobj(match=Text):
    text.set_fontfamily(font_name)

fig.canvas.draw()
display(fig)

print("한글 폰트 적용 완료:", font_name)

In [ ]:
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

root = ET.parse(
    BASE / sam_trial_case["annotation_path"]
).getroot()

x1, y1, x2, y2 = sam_trial_box.astype(int)

# 대상 주변을 넓게 표시
margin = max(x2 - x1, y2 - y1) // 2
left = max(0, x1 - margin)
top = max(0, y1 - margin)
right = min(sam_trial_image.shape[1], x2 + margin)
bottom = min(sam_trial_image.shape[0], y2 + margin)

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(sam_trial_image)
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)

for index, obj in enumerate(root.findall("object")):
    bx1, by1, bx2, by2 = [
        int(obj.findtext(f"bndbox/{key}"))
        for key in ["xmin", "ymin", "xmax", "ymax"]
    ]

    if bx2 <= left or bx1 >= right or by2 <= top or by1 >= bottom:
        continue

    selected = index == sam_trial_object_index
    color = "red" if selected else "deepskyblue"

    ax.add_patch(Rectangle(
        (bx1, by1), bx2 - bx1, by2 - by1,
        fill=False,
        edgecolor=color,
        linewidth=2 if selected else 1,
    ))

    ax.text(
        bx1, by1 - 3,
        f"{index}: {obj.findtext('name')}",
        color=color,
        fontsize=10,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
    )

ax.set_title("빨강: 대상 / 파랑: 주변 염색체의 정답 상자")
ax.set_xlabel("원본 가로 좌표")
ax.set_ylabel("원본 세로 좌표")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# 현재 원본 사진 1055781.jpg를 보고 고른 시험 좌표
assert sam_trial_case["filename"] == "1055781.jpg"

# 앞 두 점: 대상 4번으로 보이는 영역
# 뒤 두 점: 가로 방향의 주변 염색체로 보이는 영역
sam_point_coords = np.array([
    [186, 225],
    [163, 309],
    [158, 259],
    [213, 275],
], dtype=np.float32)

sam_point_labels = np.array([1, 1, 0, 0], dtype=np.int32)

assert (sam_point_coords[:, 0] >= 0).all()
assert (sam_point_coords[:, 0] < sam_trial_image.shape[1]).all()
assert (sam_point_coords[:, 1] >= 0).all()
assert (sam_point_coords[:, 1] < sam_trial_image.shape[0]).all()

# 같은 원본에 상자와 점을 함께 제공
with torch.inference_mode():
    sam_predictor.set_image(sam_trial_image)

    sam_point_masks, sam_point_scores, _ = sam_predictor.predict(
        point_coords=sam_point_coords,
        point_labels=sam_point_labels,
        box=sam_trial_box,
        multimask_output=True,
    )

# 주변 맥락을 조금 넓게 표시
x1, y1, x2, y2 = sam_trial_box.astype(int)
margin = 40

left = max(0, x1 - margin)
top = max(0, y1 - margin)
right = min(sam_trial_image.shape[1], x2 + margin)
bottom = min(sam_trial_image.shape[0], y2 + margin)

fig, axes = plt.subplots(1, 4, figsize=(16, 6))

axes[0].imshow(sam_trial_image)
axes[0].add_patch(Rectangle(
    (x1, y1), x2 - x1, y2 - y1,
    fill=False, edgecolor="red", linewidth=1.5,
))

positive = sam_point_labels == 1
negative = sam_point_labels == 0

axes[0].scatter(
    sam_point_coords[positive, 0],
    sam_point_coords[positive, 1],
    c="lime", marker="*", s=160,
    edgecolors="black", label="포함할 점",
)
axes[0].scatter(
    sam_point_coords[negative, 0],
    sam_point_coords[negative, 1],
    c="orange", marker="x", s=100,
    linewidths=2, label="제외할 점",
)
axes[0].legend(fontsize=9)
axes[0].set_title("원본과 사람이 지정한 점")

for index, (mask, score) in enumerate(
    zip(sam_point_masks, sam_point_scores)
):
    cleaned = sam_trial_image.copy()
    cleaned[~mask] = 255

    axes[index + 1].imshow(cleaned)
    axes[index + 1].set_title(
        f"점 추가 후보 {index + 1}\n"
        f"SAM 점수 {float(score):.3f}"
    )

for ax in axes:
    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("상자 + 포함점 2개 + 제외점 2개 시험 완료")
print("사람이 지정한 점을 사용한 반자동 결과야.")
print("파일 변경·재학습 없음")